# 11_MULTI-CHECKPOINT_LOGIT-POOL — Heterogeneous Checkpoint Ensemble with Stable Logit Pooling

**Goal.** Notebook 10 reached 403/450 on the held-out test split (accuracy 0.8956); the ≥ 90% target needs 405/450. Its 47 residual test errors are 28-high-confidence — threshold moves trade false positives for false negatives roughly 1:1 and calibration is provably worth zero — so the remaining rows must come from **ranking quality**. Seed diversity of one checkpoint is a measured ceiling (notebook 08 regressed; notebook 10 gained +0.67pp and stopped), but the repository's own CPU pilot found that a heterogeneous 4-model set leaves only 42/450 validation rows wrong where the anchor alone leaves 61: roughly 19 recoverable rows. This notebook pools, by notebook 10's proven mean-log-odds rule at a prespecified 0.50 cutoff, three-seed pools of **five checkpoints spanning fine-tuning lineage, pretraining corpus, architecture/tokenizer, and scale**, anchored on notebook 10's exact recipe so its floor is preserved.

**Component roster** (all repo ids verified live on the Hugging Face Hub 2026-08-08):

| Component | Checkpoint | Diversity axis | Head |
|---|---|---|---|
| A anchor | `cardiffnlp/twitter-roberta-base-hate-latest` | notebook 10's exact winner | pretrained 2-class kept |
| B dynabench | `facebook/roberta-hate-speech-dynabench-r4-target` | adversarially-collected implicit hate | pretrained 2-class + orientation probe |
| C hatebert | `GroNLP/hateBERT` | Reddit-abuse MLM corpus, WordPiece | fresh 2-class |
| D deberta | `microsoft/deberta-v3-base` | architecture + SentencePiece tokenizer | fresh 2-class |
| E large | `cardiffnlp/twitter-roberta-large-hate-latest` | scale on the matched domain | fresh 2-class (its pretrained head is an 8-class hate taxonomy) |
| F anchor+FGM | same as A, FGM adversarial training | decision-boundary smoothing | pretrained 2-class kept |

**Prespecified decision rule.** Primary = equal-weight mean of admitted components' (A–E) seed-mean logits at fixed threshold 0.50. Exactly four challengers are evaluated once each (anchor-weight-2; top-3 components; primary ∪ F; validation-tuned stable threshold) and a challenger replaces the primary only if it gains **≥ 3 correct validation examples**. A component enters the pool only if its own pooled validation accuracy at 0.50 is ≥ 0.84 (broken-run guard). If the selected strategy trails the anchor-alone pool by ≥ 3 validation rows, the anchor-alone fallback ships and the test unlock is refused.

**Test discipline.** All selection happens on train + validation. Per-run test probabilities are computed inference-only and stored; **no test metric is computed until every choice is locked**, and only if the selected strategy's validation accuracy reaches **392/450 (0.8711)** — the level that, through the measured +2.83pp validation→test offset, projects ≈ 0.90 test. (`enforce_validation_gate` in CONFIG documents this gate; it exists to keep a one-shot test unlock from being spent on an expected miss. Validation runs ~2–3pp harder than test for every technique on record — do not read validation numbers against 0.90 directly.) Declared validation budget: 8 screening + 6 admission + 5 decision-layer = **19 scored looks**, recorded in `metadata.json`.

**Expected outcome (stated before the run).** Expected primary pooled validation accuracy 0.868–0.880. Even a test result above 0.90 is far below the Holm-corrected significance bar at family size 10; the realistic best verdict is *reaches the project's 90% target; NOT statistically distinguishable from 07_TWITTER-ROBERTA_FINE-TUNE at n=450*. `tools/compare_techniques.py` is the sole verdict authority; this notebook reports numbers and issues no verdict.

**Run setup (Kaggle).** Accelerator: single T4 or P100 (16 GB). Internet: ON (checkpoints download from the Hub, ~3.6 GB). Attach the notebook-00 output (`research_foundation/`: `processed_dataset.csv`, `split_assignments.csv`, `dataset_manifest.json`) as an input; files are auto-discovered under `/kaggle/input`. Nominal runtime ≈ 2.5 h; worst case ≈ 4.5 h with automatic scope degradation long before the 9 h session cap. Outputs land in `/kaggle/working/experiments/11_MULTI-CHECKPOINT_LOGIT-POOL/` and are zipped by the final cell; the sanitized `probs/` artifacts are the only files intended for the repository.

**Responsible use.** This is a research benchmark on a fixed split of a curated dataset. The models are not moderation systems; predictions must not be treated as judgments about people or used for enforcement, and strong benchmark metrics are not deployment evidence (see `docs/RESPONSIBLE_USE.md`).


In [ ]:
# ============================================================
# 1. Imports and environment
# ============================================================

import os
import gc
import re
import json
import math
import time
import html
import copy
import shutil
import hashlib
import random
import platform
import warnings
import unicodedata
import subprocess
import sys
import datetime as dt
from pathlib import Path
from itertools import product
from contextlib import nullcontext


def ensure_package(import_name: str, pip_name: str) -> None:
    """Install a missing dependency in-notebook (Kaggle images vary)."""
    try:
        __import__(import_name)
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pip_name])


# DeBERTa-v3's tokenizer requires sentencepiece (+ protobuf for its converter).
ensure_package("sentencepiece", "sentencepiece")
ensure_package("google.protobuf", "protobuf")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    brier_score_loss,
    roc_curve,
    precision_recall_curve,
)
from sklearn.calibration import calibration_curve

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

import joblib

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 250)
pd.set_option("display.max_colwidth", 140)

RUN_START_TIME = time.time()

print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# 2. Central configuration
# ============================================================

CONFIG = {
    "project_name": "extremism_text_classification",
    "technique_name": "11_MULTI-CHECKPOINT_LOGIT-POOL",
    "model_family": "heterogeneous_checkpoint_logit_pooled_transformer_ensemble",
    "feature_family": "contextual_transformer_logit_probability_pooling",
    "dataset_version": "extremism_dataset_clean_v1",
    "split_version": "split_v1_stratified_70_15_15_seed30",
    "random_seed": 30,
    "final_seeds": [17, 30, 73],

    # Canonical files from the dataset/split notebook (auto-discovered when None).
    "processed_dataset_path": None,
    "split_assignments_path": None,
    "dataset_manifest_path": None,
    "verify_input_hashes_against_manifest": True,
    # Hard assert before any training: (negative, positive) counts of the frozen
    # assignment split_v1_stratified_70_15_15_seed30.
    "expected_split_label_counts": {
        "train": (1309, 790), "validation": (281, 169), "test": (280, 170),
    },

    "id_col": "row_id",
    "text_col": "text",
    "label_col": "label",
    "split_col": "split",
    "positive_label": 1,
    "text_normalization_mode": "minimal_twitter",  # verified no-op on this corpus; kept for comparability
    "max_length": 192,

    "output_root": "/kaggle/working/experiments",
    "overwrite_output_dir": True,

    # ------------------------------------------------------------------
    # Component roster. Diversity axes: fine-tuning lineage (A/B), pretraining
    # corpus (C), architecture + tokenizer (D), scale (E), training technique (F).
    # F is excluded from the prespecified primary pool and can enter only through
    # challenger C3 under the >=3-validation-row gate.
    # ------------------------------------------------------------------
    "components": {
        "A_anchor": {
            "checkpoint": "cardiffnlp/twitter-roberta-base-hate-latest",
            "download_fallback": None,
            "on_download_failure": "hard_abort",   # the design is meaningless without its floor
            "head_policy": "pretrained",
            "orientation_probe": False,
            "learning_rate": 3e-5,                  # nb10's screened winner; not re-screened
            "lr_candidates": None,
            "num_epochs": 3,
            "batch_size": 8,
            "gradient_accumulation_steps": 2,
            "precision": "fp16_scaler",
            "gradient_checkpointing": False,
            "fgm": False,
            "seeds": [17, 30, 73],
            "in_primary_pool": True,
            "droppable_on_time_budget": False,
        },
        "B_dynabench": {
            "checkpoint": "facebook/roberta-hate-speech-dynabench-r4-target",
            "download_fallback": "tomh/toxigen_roberta",
            "fallback_head_policy": "fresh",
            "on_download_failure": "try_fallback_then_drop",
            "head_policy": "pretrained",            # id2label {0: nothate, 1: hate} verified
            "orientation_probe": True,
            "orientation_probe_rows": 512,
            "lr_candidates": [2e-5, 3e-5],
            "num_epochs": 3,
            "batch_size": 8,
            "gradient_accumulation_steps": 2,
            "precision": "fp16_scaler",
            "gradient_checkpointing": False,
            "fgm": False,
            "seeds": [17, 30, 73],
            "in_primary_pool": True,
            "droppable_on_time_budget": False,
        },
        "C_hatebert": {
            "checkpoint": "GroNLP/hateBERT",
            "download_fallback": None,
            "on_download_failure": "drop",
            "head_policy": "fresh",
            "orientation_probe": False,
            "lr_candidates": [2e-5, 3e-5],
            "num_epochs": 3,
            "batch_size": 8,
            "gradient_accumulation_steps": 2,
            "precision": "fp16_scaler",
            "gradient_checkpointing": False,
            "fgm": False,
            "seeds": [17, 30, 73],
            "in_primary_pool": True,
            "droppable_on_time_budget": False,
        },
        "D_deberta": {
            "checkpoint": "microsoft/deberta-v3-base",
            "download_fallback": None,
            "on_download_failure": "drop",
            "head_policy": "fresh",
            "orientation_probe": False,
            "lr_candidates": [2e-5, 3e-5],
            "num_epochs": 3,
            "batch_size": 8,
            "gradient_accumulation_steps": 2,
            "precision": "fp16_nan_guard_then_fp32_restart",  # known fp16-fragile family
            "gradient_checkpointing": False,
            "fgm": False,
            "seeds": [17, 30, 73],
            "in_primary_pool": True,
            "droppable_on_time_budget": False,
        },
        "E_twitter_large": {
            "checkpoint": "cardiffnlp/twitter-roberta-large-hate-latest",
            "download_fallback": "cardiffnlp/twitter-roberta-large-2022-154m",
            "fallback_head_policy": "fresh",
            "on_download_failure": "try_fallback_then_drop",
            # The pretrained head is a verified 8-class hate taxonomy
            # (hate_gender ... not_hate): it cannot be kept for binary use.
            "head_policy": "fresh",
            "orientation_probe": False,
            "lr_candidates": [1e-5, 2e-5],          # large models degenerate at base-model LRs
            "num_epochs": 3,
            "batch_size": 8,
            "gradient_accumulation_steps": 2,
            "oom_ladder": ["gradient_checkpointing_on", "batch_size_4_accum_4"],
            "precision": "fp16_scaler",
            "gradient_checkpointing": False,
            "fgm": False,
            "seeds": [17, 30, 73],
            "in_primary_pool": True,
            "droppable_on_time_budget": True,
            "drop_priority": 2,
        },
        "F_anchor_fgm": {
            "checkpoint": "cardiffnlp/twitter-roberta-base-hate-latest",
            "download_fallback": None,
            "on_download_failure": "drop",
            "head_policy": "pretrained",
            "orientation_probe": False,
            "learning_rate": 3e-5,
            "lr_candidates": None,
            "num_epochs": 3,
            "batch_size": 8,
            "gradient_accumulation_steps": 2,
            "precision": "fp16_scaler",
            "gradient_checkpointing": False,
            "fgm": True,
            "seeds": [17, 30, 73],
            "in_primary_pool": False,
            "enters_via_challenger": "C3_pool_plus_fgm",
            "droppable_on_time_budget": True,
            "drop_priority": 1,
        },
    },

    # Head policy by checkpoint id (covers download fallbacks too).
    "checkpoint_head_policy": {
        "cardiffnlp/twitter-roberta-base-hate-latest": "pretrained",
        "facebook/roberta-hate-speech-dynabench-r4-target": "pretrained",
        "GroNLP/hateBERT": "fresh",
        "microsoft/deberta-v3-base": "fresh",
        "cardiffnlp/twitter-roberta-large-hate-latest": "fresh",
        "cardiffnlp/twitter-roberta-large-2022-154m": "fresh",
        "tomh/toxigen_roberta": "fresh",
    },
    "default_head_policy": "fresh",
    # A 'pretrained' policy is only legal when the source head is binary; declared
    # label counts are verified from each checkpoint's config at load time.
    "checkpoint_pretrained_head_labels": {
        "cardiffnlp/twitter-roberta-base-hate-latest": 2,
        "facebook/roberta-hate-speech-dynabench-r4-target": 2,
    },

    # Shared fixed recipe (no tuning): nb10's thrice-confirmed winner.
    "transformer_training": {
        "weight_decay": 0.01,
        "label_smoothing": 0.0,
        "class_weight": None,
        "gradient_accumulation_steps": 2,
        "warmup_ratio": 0.10,
        "max_grad_norm": 1.0,
        "drop_last": False,
        "num_workers": 2,
        "use_amp_if_cuda": True,
        "amp_dtype": "float16",                    # explicit: no bf16-on-T4 autodetect
        "force_fp32_master_weights": True,
        "early_stopping_patience_epochs": 1,
        "metric_for_best_epoch": "accuracy_at_0_50",  # the monitor measures the deployed rule
        "nan_guard_events_before_fp32_restart": 2,
        "fp32_restart_max_attempts": 1,
        "save_all_seed_models": False,
    },

    "fgm": {
        "epsilon": 1.0,                             # prespecified; no tuning
        "norm": "l2_global",
        # The attack direction g/||g|| is invariant to the GradScaler multiplier.
    },

    "screening": {
        "screen_seed": 30,
        "rank_metric": "validation_accuracy_at_fixed_0_50",
        "tie_break": ["balanced_accuracy", "positive_f1", "pr_auc"],
        "reuse_winning_screen_run_as_final_seed_30_run": True,
    },

    "pooling": {
        "within_component": "mean_of_seed_logits",
        "across_components": "equal_weight_mean_of_component_mean_logits",
        "logit_clip_eps": 1e-6,
        "sigmoid_clip": 40.0,
        "admission_floor_val_accuracy_at_0_50": 0.84,   # broken-run guard, not a tuner
        "min_seeds_for_admission": 2,
    },

    "selection": {
        "prespecified_primary": "equal_weight_logit_mean_admitted_components_A_to_E__fixed_0_50",
        "prespecified_primary_threshold": 0.50,
        "challengers": [
            {"id": "C1", "rule": "anchor_weight_2_logit_mean__fixed_0_50"},
            {"id": "C2", "rule": "top3_components_by_pooled_val_accuracy_logit_mean__fixed_0_50"},
            {"id": "C3_pool_plus_fgm", "rule": "primary_pool_plus_F_anchor_fgm__fixed_0_50"},
            {"id": "C4", "rule": "primary_pool__validation_tuned_stable_threshold"},
        ],
        "minimum_gain_correct_examples_over_primary": 3,
        "fallback_to_anchor_only_if_primary_trails_by_rows": 3,
        "anchor_only_fallback_refuses_test_unlock": True,
    },

    "threshold_selection": {
        "metric": "accuracy",
        "threshold_grid": [round(x, 3) for x in np.linspace(0.05, 0.95, 181)],
        # The stability machinery is ON for the one tuned-threshold challenger
        # (nb10 shipped it disabled; tolerance 0 / repeats 0 degenerated to a
        # fragile exact argmax).
        "near_optimal_tolerance_correct_examples": 1,
        "bootstrap_repeats": 200,
    },

    "test_unlock_gate": {
        # 392/450 projects to ~0.899-0.902 test through the measured +2.83pp
        # (sd 0.38) validation->test offset: the minimum at which spending a
        # one-shot, Holm-bar-raising unlock is rational for a >=0.90 goal.
        "enforce_validation_gate": True,
        "min_val_correct_of_450": 392,
        "refuse_unlock_if_selected_is_anchor_only_fallback": True,
    },

    "validation_budget_declared": {
        "screening_scores": 8,
        "admission_scores": 6,
        "decision_layer_scores": 5,
        "total_scored_looks": 19,
        "per_epoch_early_stopping_peeks": "<=3 per training run across <=22 runs",
    },

    "time_budget": {
        "session_cap_seconds": 32400,
        "stop_launching_E_final_runs_after_seconds": 18000,
        "drop_F_after_seconds": 23400,
        "skip_xai_and_trim_error_analysis_after_seconds": 27000,
    },

    "probs_export": {
        "required_columns": ["row_id", "split", "y_true", "y_prob"],
        "export_selected_strategy": True,
        "export_per_component_pooled": True,
        "export_per_run": True,
        "export_even_if_gate_fails": True,   # test probabilities are inference-only artifacts
    },

    "statistics": {
        "family_size_for_holm": 10,   # 01-07, 08, 10, plus this run; confirm vs ledger at commit
        "comparator": "07_TWITTER-ROBERTA_FINE-TUNE",
        "verdict_authority": "tools/compare_techniques.py",
        "mcnemar_unadjusted_min_b_minus_c": 14,
        "mcnemar_holm_min_b_minus_c": 20,
        "mandated_inconclusive_phrase": (
            "reaches the project's 90% target; NOT statistically distinguishable "
            "from 07_TWITTER-ROBERTA_FINE-TUNE at n=450"
        ),
    },

    "explainability": {
        "representative_component": "A_anchor",
        "representative_seed": 30,
        "top_n_global": 40,
        "top_n_local_tokens": 15,
        "local_examples_per_bucket": 8,
        "max_global_attribution_examples": 150,
    },

    "error_analysis": {
        "include_text_preview": False,   # row_id-keyed only; sanitized outputs
        "examples_per_bucket": 30,
        "near_threshold_margin": 0.05,
        "high_confidence_positive": 0.90,
        "high_confidence_negative": 0.10,
    },
}

TECHNIQUE_NAME = CONFIG["technique_name"]
RUN_ID = f"{TECHNIQUE_NAME}_{dt.datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = Path(CONFIG["output_root"]) / TECHNIQUE_NAME

if CONFIG["overwrite_output_dir"] and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for subdir in [
    OUTPUT_DIR,
    OUTPUT_DIR / "ablation",
    OUTPUT_DIR / "auxiliary",
    OUTPUT_DIR / "model_artifacts",
    OUTPUT_DIR / "plots",
    OUTPUT_DIR / "interpretability",
    OUTPUT_DIR / "error_analysis",
    OUTPUT_DIR / "training_logs",
    OUTPUT_DIR / "data_quality",
    OUTPUT_DIR / "probs",
]:
    subdir.mkdir(parents=True, exist_ok=True)

print("Run ID:", RUN_ID)
print("Output directory:", OUTPUT_DIR)
print(json.dumps({k: v for k, v in CONFIG.items() if k != "components"}, indent=2, default=str))
print("Components:", list(CONFIG["components"].keys()))


In [ ]:
# ============================================================
# 3. Reproducibility, serialization, and file helpers
# ============================================================

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # benchmark=True is faster; all seeds and selected configurations are still recorded.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return None if math.isnan(float(obj)) else float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if torch.is_tensor(obj):
        return obj.detach().cpu().tolist()
    try:
        if pd.isna(obj) and not isinstance(obj, (list, tuple, dict, np.ndarray)):
            return None
    except Exception:
        pass
    return obj


def save_json(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(make_json_safe(obj), f, indent=2, ensure_ascii=False)


def sha256_file(path: Path, n_chars: int = 16) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()[:n_chars]


def sha256_text(text: str, n_chars: int = 16) -> str:
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()[:n_chars]


def find_file_by_name(file_name: str, search_roots=None) -> Path:
    # On Kaggle only /kaggle/input is searched: the working directory (which is
    # also the notebook's cwd) could shadow the canonical attached input with
    # the run's own outputs on a rerun. The cwd fallback exists for local runs.
    if search_roots is None:
        kaggle_input = Path("/kaggle/input")
        search_roots = [kaggle_input] if kaggle_input.exists() else [Path(".")]
    matches = []
    for root in search_roots:
        if root.exists():
            matches.extend(root.rglob(file_name))
    matches = sorted(set(matches), key=lambda p: (len(str(p)), str(p)))
    if not matches:
        raise FileNotFoundError(
            f"Could not find {file_name}. Set its exact path in CONFIG or attach it to the notebook."
        )
    if len(matches) > 1:
        print(f"Multiple matches found for {file_name}; using the shortest path:")
        for match in matches[:10]:
            print(" -", match)
    return matches[0]


def build_param_grid(grid: dict):
    keys = list(grid.keys())
    values = [grid[k] for k in keys]
    return [dict(zip(keys, combination)) for combination in product(*values)]


def sanitize_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(value)).strip("_")


def elapsed_hours() -> float:
    return (time.time() - RUN_START_TIME) / 3600.0


set_global_seed(CONFIG["random_seed"])
save_json(CONFIG, OUTPUT_DIR / "config.json")
print("Seed set to", CONFIG["random_seed"])


In [ ]:
# ============================================================
# 4. Load canonical dataset, split assignments, and manifest
# ============================================================

processed_dataset_path = (
    Path(CONFIG["processed_dataset_path"])
    if CONFIG["processed_dataset_path"] is not None
    else find_file_by_name("processed_dataset.csv")
)
split_assignments_path = (
    Path(CONFIG["split_assignments_path"])
    if CONFIG["split_assignments_path"] is not None
    else find_file_by_name("split_assignments.csv")
)
dataset_manifest_path = (
    Path(CONFIG["dataset_manifest_path"])
    if CONFIG["dataset_manifest_path"] is not None
    else find_file_by_name("dataset_manifest.json")
)

processed_df = pd.read_csv(processed_dataset_path)
splits = pd.read_csv(split_assignments_path)
with open(dataset_manifest_path, "r", encoding="utf-8") as f:
    dataset_manifest = json.load(f)

# The attached foundation output is the only trusted source. The manifest
# carries no per-file hashes, so consistency is checked against the counts it
# does record (a stale or shadowed input fails here or at the split-label
# assert in the next cell); observed hashes are recorded for the metadata.
input_file_hashes = {
    "processed_dataset.csv": sha256_file(processed_dataset_path),
    "split_assignments.csv": sha256_file(split_assignments_path),
}
if CONFIG["verify_input_hashes_against_manifest"]:
    manifest_rows = dataset_manifest.get("processed_rows")
    if manifest_rows is not None and int(manifest_rows) != len(processed_df):
        raise RuntimeError(
            f"processed_dataset.csv has {len(processed_df)} rows but the manifest "
            f"records {manifest_rows}. The attached research_foundation output is "
            "stale or shadowed."
        )
    manifest_split_counts = dataset_manifest.get("split_counts_actual") or {}
    observed_split_counts = splits["split"].value_counts().to_dict()
    for split_name, expected_count in manifest_split_counts.items():
        observed_count = int(observed_split_counts.get(split_name, 0))
        if observed_count != int(expected_count):
            raise RuntimeError(
                f"split_assignments.csv has {observed_count} {split_name!r} rows but "
                f"the manifest records {expected_count}. The attached "
                "research_foundation output is stale or shadowed."
            )
    manifest_labels = dataset_manifest.get("label_counts") or {}
    if manifest_labels:
        observed_labels = processed_df["label"].astype(int).value_counts().to_dict()
        for label_value, expected_count in manifest_labels.items():
            if int(observed_labels.get(int(label_value), 0)) != int(expected_count):
                raise RuntimeError(
                    f"Label counts disagree with the manifest for class {label_value}. "
                    "The attached research_foundation output is stale or shadowed."
                )
    print("Manifest consistency verified (rows, split counts, label counts).")

print("Processed dataset path:", processed_dataset_path)
print("Split assignments path:", split_assignments_path)
print("Dataset manifest path:", dataset_manifest_path)
print("Processed dataset shape:", processed_df.shape)
print("Split assignment shape:", splits.shape)
print("Input hashes:", input_file_hashes)
# Leakage hygiene: never display() a text-bearing frame; committed notebooks are
# scanned for dataset text in saved outputs. Print schema only.
print("Processed columns:", sorted(processed_df.columns.tolist()))
print("Split columns:", sorted(splits.columns.tolist()))


In [ ]:
# ============================================================
# 5. Validate schema, hard-assert split integrity, and normalize text
# ============================================================

id_col = CONFIG["id_col"]
text_col = CONFIG["text_col"]
label_col = CONFIG["label_col"]
split_col = CONFIG["split_col"]
MODEL_TEXT_COL = "__model_input_text"
SOURCE_TEXT_COL = "__source_text"

required_processed_cols = {id_col, text_col, label_col}
required_split_cols = {id_col, split_col}
missing_processed = required_processed_cols - set(processed_df.columns)
missing_splits = required_split_cols - set(splits.columns)
if missing_processed:
    raise ValueError(f"Processed dataset is missing required columns: {missing_processed}")
if missing_splits:
    raise ValueError(f"Split assignment file is missing required columns: {missing_splits}")
if processed_df[id_col].duplicated().any():
    raise ValueError("processed_dataset.csv contains duplicate row_id values.")
if splits[id_col].duplicated().any():
    raise ValueError("split_assignments.csv contains duplicate row_id values.")

working_df = processed_df.copy()
# There is no less-processed text variant anywhere in the foundation output
# (verified against notebook 00: `text` is whitespace-collapsed Original_Message).
working_df[SOURCE_TEXT_COL] = working_df[text_col].fillna("").astype(str)

URL_RE = re.compile(r"(?i)\b(?:https?://|www\.)\S+|\bhttp\S*")
USER_RE = re.compile(r"(?<!\w)@[A-Za-z0-9_]+")
WHITESPACE_RE = re.compile(r"\s+")


def normalize_model_text(text: str, mode: str) -> str:
    text = "" if pd.isna(text) else str(text)
    if mode == "identity":
        return text
    if mode != "minimal_twitter":
        raise ValueError(f"Unsupported text_normalization_mode: {mode}")
    # Kept verbatim from notebook 10 for cross-notebook comparability. On this
    # corpus (lowercase ASCII, no URLs/mentions/case/punctuation) it is a
    # verified byte-level no-op; it exists to guard against future data drift.
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    text = USER_RE.sub("@user", text)
    text = URL_RE.sub("http", text)
    text = WHITESPACE_RE.sub(" ", text).strip()
    return text


working_df[MODEL_TEXT_COL] = working_df[SOURCE_TEXT_COL].map(
    lambda value: normalize_model_text(value, CONFIG["text_normalization_mode"])
)
working_df[label_col] = working_df[label_col].astype(int)

split_merge_columns = [id_col, split_col]
for mirror_col in (label_col, "text_hash"):
    if mirror_col in splits.columns:
        split_merge_columns.append(mirror_col)
merged = working_df.merge(
    splits[split_merge_columns],
    on=id_col,
    how="left",
    validate="one_to_one",
    suffixes=("", "__from_splits"),
)
if merged[split_col].isna().any():
    missing_ids = merged.loc[merged[split_col].isna(), id_col].head().tolist()
    raise ValueError(f"Some rows are missing split assignments. Examples: {missing_ids}")

# Cross-check labels and text hashes between the two files when the split
# mirror carries them: a stale mirror must fail loudly, not train silently.
label_mirror_col = f"{label_col}__from_splits"
if label_mirror_col in merged.columns:
    label_disagreements = int(
        (merged[label_col].astype(int) != merged[label_mirror_col].astype(int)).sum()
    )
    if label_disagreements:
        raise ValueError(
            f"{label_disagreements} rows disagree on label between processed_dataset.csv "
            "and split_assignments.csv. The attached inputs are inconsistent."
        )
    merged = merged.drop(columns=[label_mirror_col])

hash_mirror_col = "text_hash__from_splits"
if hash_mirror_col in merged.columns and "text_hash" in merged.columns:
    hash_disagreements = int(
        (merged["text_hash"].astype(str) != merged[hash_mirror_col].astype(str)).sum()
    )
    if hash_disagreements:
        raise ValueError(
            f"{hash_disagreements} rows disagree on text_hash between processed_dataset.csv "
            "and split_assignments.csv. The attached inputs are inconsistent."
        )
    merged = merged.drop(columns=[hash_mirror_col])

allowed_splits = {"train", "validation", "test"}
observed_splits = set(merged[split_col].unique())
if not observed_splits.issubset(allowed_splits):
    raise ValueError(f"Unexpected split labels: {observed_splits - allowed_splits}")
if not set(merged[label_col].unique()).issubset({0, 1}):
    raise ValueError("This notebook expects binary labels encoded as 0 and 1.")

# Hard split-integrity assert BEFORE any training: the frozen assignment
# split_v1_stratified_70_15_15_seed30 has exactly these (negative, positive) counts.
for split_name, (expected_neg, expected_pos) in CONFIG["expected_split_label_counts"].items():
    split_part = merged[merged[split_col] == split_name]
    observed_neg = int((split_part[label_col] == 0).sum())
    observed_pos = int((split_part[label_col] == 1).sum())
    if (observed_neg, observed_pos) != (int(expected_neg), int(expected_pos)):
        raise RuntimeError(
            f"Split integrity failure for {split_name!r}: expected "
            f"(neg={expected_neg}, pos={expected_pos}), observed "
            f"(neg={observed_neg}, pos={observed_pos}). Refusing to train on a "
            "split that is not the frozen assignment."
        )

train_df = merged[merged[split_col] == "train"].reset_index(drop=True)
val_df = merged[merged[split_col] == "validation"].reset_index(drop=True)
test_df = merged[merged[split_col] == "test"].reset_index(drop=True)

text_source_audit = {
    "selected_source_text_column": text_col,
    "normalization_mode": CONFIG["text_normalization_mode"],
    "normalization_changed_any_row": bool(
        (working_df[MODEL_TEXT_COL] != working_df[SOURCE_TEXT_COL]).any()
    ),
    "note": (
        "processed_dataset.text is the least-processed text available; upstream "
        "cleaning (case/punctuation/URL removal, slang expansion) happened before "
        "this repository's dataset was created and is unrecoverable."
    ),
}
save_json(text_source_audit, OUTPUT_DIR / "data_quality" / "text_source_audit.json")

print("Split integrity verified:", {
    "train": len(train_df), "validation": len(val_df), "test": len(test_df)
})
print("Normalization changed any row:", text_source_audit["normalization_changed_any_row"])

split_audit = merged.groupby([split_col, label_col]).size().reset_index(name="count")
split_audit["rate_within_split"] = split_audit.groupby(split_col)["count"].transform(
    lambda s: s / s.sum()
)
split_audit.to_csv(OUTPUT_DIR / "split_label_distribution_used.csv", index=False)
display(split_audit)


In [ ]:
# ============================================================
# 6. Data-quality and leakage audit (hash-level only)
# ============================================================

quality_df = merged[[id_col, split_col, label_col, MODEL_TEXT_COL]].copy()
quality_df["model_text_hash"] = quality_df[MODEL_TEXT_COL].map(sha256_text)
quality_df["model_text_length"] = quality_df[MODEL_TEXT_COL].str.len()

empty_model_text = quality_df[quality_df["model_text_length"] == 0]
duplicate_groups = (
    quality_df.groupby("model_text_hash", as_index=False)
    .agg(
        row_count=(id_col, "size"),
        split_count=(split_col, "nunique"),
        label_count=(label_col, "nunique"),
        splits=(split_col, lambda s: "|".join(sorted(set(map(str, s))))),
        labels=(label_col, lambda s: "|".join(sorted(set(map(str, s))))),
    )
)
cross_split_duplicates = duplicate_groups[
    (duplicate_groups["row_count"] > 1) & (duplicate_groups["split_count"] > 1)
]
conflicting_label_duplicates = duplicate_groups[
    (duplicate_groups["row_count"] > 1) & (duplicate_groups["label_count"] > 1)
]

# Hash-only outputs: row ids and digests, never text.
cross_split_duplicates.to_csv(
    OUTPUT_DIR / "data_quality" / "cross_split_duplicate_text_groups.csv", index=False
)
conflicting_label_duplicates.to_csv(
    OUTPUT_DIR / "data_quality" / "conflicting_label_duplicate_text_groups.csv", index=False
)
empty_model_text[[id_col, split_col, label_col, "model_text_hash"]].to_csv(
    OUTPUT_DIR / "data_quality" / "empty_model_text_rows.csv", index=False
)

quality_summary = {
    "empty_model_text_rows": int(len(empty_model_text)),
    "cross_split_duplicate_text_groups": int(len(cross_split_duplicates)),
    "conflicting_label_duplicate_text_groups": int(len(conflicting_label_duplicates)),
    "note": (
        "Exact-duplicate audit only. Near-duplicate template families are known "
        "to span splits in this dataset; they are a property of the frozen "
        "assignment and are not re-partitioned here."
    ),
}
save_json(quality_summary, OUTPUT_DIR / "data_quality" / "data_quality_summary.json")
print(json.dumps(quality_summary, indent=2))


In [ ]:
# ============================================================
# 7. Metrics, threshold selection, pooling, and export utilities
# ============================================================

def safe_roc_auc(y_true, y_prob):
    try:
        return np.nan if len(np.unique(y_true)) < 2 else roc_auc_score(y_true, y_prob)
    except Exception:
        return np.nan


def safe_pr_auc(y_true, y_prob):
    try:
        return np.nan if len(np.unique(y_true)) < 2 else average_precision_score(y_true, y_prob)
    except Exception:
        return np.nan


def compute_binary_metrics(y_true, y_prob, threshold, positive_label=1):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= float(threshold)).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "threshold": float(threshold),
        "support": int(len(y_true)),
        "positive_support": int((y_true == positive_label).sum()),
        "negative_support": int((y_true != positive_label).sum()),
        "positive_rate": float((y_true == positive_label).mean()),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision_weighted": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "recall_weighted": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "positive_precision": float(precision_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
        "positive_recall": float(recall_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
        "positive_f1": float(f1_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
        "roc_auc": float(safe_roc_auc(y_true, y_prob)),
        "pr_auc": float(safe_pr_auc(y_true, y_prob)),
        "brier_score": float(brier_score_loss(y_true, y_prob)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "false_positive_rate": float(fp / (fp + tn)) if (fp + tn) else np.nan,
        "false_negative_rate": float(fn / (fn + tp)) if (fn + tp) else np.nan,
    }


def build_threshold_candidates(y_prob, configured_thresholds=None):
    """Return all decision-change intervals plus configured reference cutoffs.

    For predictions defined by p >= threshold, accuracy can change only when the
    threshold crosses a predicted probability. Midpoints between adjacent unique
    probabilities represent every distinct prediction set without relying on a
    coarse grid. Configured thresholds and 0.50 are retained for comparability.
    """
    probability = np.asarray(y_prob, dtype=float)
    probability = probability[np.isfinite(probability)]
    if probability.size == 0:
        raise ValueError("No finite probabilities were supplied for threshold selection.")

    unique_probability = np.unique(np.clip(probability, 0.0, 1.0))
    if unique_probability.size > 1:
        midpoint_candidates = (unique_probability[:-1] + unique_probability[1:]) / 2.0
    else:
        midpoint_candidates = unique_probability.copy()

    configured = np.asarray(list(configured_thresholds or []) + [0.50], dtype=float)
    candidates = np.unique(np.concatenate([
        midpoint_candidates,
        configured,
        np.array([0.0, 1.0], dtype=float),
    ]))
    return candidates[(candidates >= 0.0) & (candidates <= 1.0)]


def select_threshold(y_true, y_prob, thresholds, metric="accuracy"):
    candidate_thresholds = build_threshold_candidates(y_prob, thresholds)
    rows = [
        compute_binary_metrics(y_true, y_prob, threshold, CONFIG["positive_label"])
        for threshold in candidate_thresholds
    ]
    threshold_df = pd.DataFrame(rows)
    threshold_df["distance_to_0_50"] = (threshold_df["threshold"] - 0.50).abs()
    # Accuracy is primary. Macro/balanced metrics prevent a majority-class tie-break;
    # the final distance term prefers the least extreme of exactly tied cutoffs.
    threshold_df = threshold_df.sort_values(
        by=[metric, "balanced_accuracy", "f1_macro", "positive_f1", "distance_to_0_50"],
        ascending=[False, False, False, False, True],
    ).reset_index(drop=True)
    return float(threshold_df.iloc[0]["threshold"]), threshold_df


def _fast_accuracy_by_threshold(y_true, y_prob, thresholds):
    y = np.asarray(y_true, dtype=int)[:, None]
    p = np.asarray(y_prob, dtype=float)[:, None]
    t = np.asarray(thresholds, dtype=float)[None, :]
    return ((p >= t).astype(int) == y).mean(axis=0)


def _stratified_bootstrap_indices(y_true, rng):
    y = np.asarray(y_true, dtype=int)
    sampled_parts = []
    for label in np.unique(y):
        label_indices = np.flatnonzero(y == label)
        sampled_parts.append(rng.choice(label_indices, size=len(label_indices), replace=True))
    sampled = np.concatenate(sampled_parts)
    rng.shuffle(sampled)
    return sampled


def select_stable_threshold(
    y_true,
    y_prob,
    thresholds,
    metric="accuracy",
    tolerance_correct_examples=1,
    bootstrap_repeats=200,
    random_seed=30,
):
    """Select a validation threshold from a near-optimal, bootstrap-stable region.

    The full sweep is saved for comparability. The final threshold is not
    required to be the single sharp maximum when neighboring thresholds differ
    by only one validation example — that maximum is noise on 450 rows.
    """
    base_threshold, threshold_df = select_threshold(y_true, y_prob, thresholds, metric=metric)
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    thresholds = np.sort(threshold_df["threshold"].astype(float).unique())

    best_metric = float(threshold_df[metric].max())
    tolerance = float(tolerance_correct_examples) / max(1, len(y_true))
    eligible = threshold_df[threshold_df[metric] >= best_metric - tolerance - 1e-12].copy()

    rng = np.random.default_rng(int(random_seed))
    bootstrap_choices = []
    for _ in range(max(0, int(bootstrap_repeats))):
        sample_idx = _stratified_bootstrap_indices(y_true, rng)
        sampled_accuracy = _fast_accuracy_by_threshold(
            y_true[sample_idx], y_prob[sample_idx], thresholds
        )
        best_indices = np.flatnonzero(sampled_accuracy >= sampled_accuracy.max() - 1e-12)
        bootstrap_choices.append(float(np.median(thresholds[best_indices])))

    bootstrap_anchor = (
        float(np.median(bootstrap_choices)) if bootstrap_choices else float(base_threshold)
    )
    eligible["distance_to_bootstrap_anchor"] = (eligible["threshold"] - bootstrap_anchor).abs()
    eligible["distance_to_0_50"] = (eligible["threshold"] - 0.50).abs()
    eligible = eligible.sort_values(
        [
            metric, "balanced_accuracy", "positive_f1", "f1_macro",
            "distance_to_bootstrap_anchor", "brier_score", "distance_to_0_50",
        ],
        ascending=[False, False, False, False, True, True, True],
    ).reset_index(drop=True)

    selected = float(eligible.iloc[0]["threshold"])
    threshold_df["selected_by_stable_rule"] = np.isclose(
        threshold_df["threshold"].astype(float), selected
    )
    threshold_df["bootstrap_anchor_threshold"] = bootstrap_anchor
    threshold_df["near_optimal_metric_floor"] = best_metric - tolerance

    stability = {
        "base_best_threshold": float(base_threshold),
        "selected_threshold": selected,
        "bootstrap_anchor_threshold": bootstrap_anchor,
        "best_validation_metric": best_metric,
        "near_optimal_tolerance": tolerance,
        "eligible_threshold_count": int(len(eligible)),
        "bootstrap_repeats": int(max(0, int(bootstrap_repeats))),
    }
    return selected, threshold_df, stability


def probability_to_logit(probability, eps=1e-6):
    p = np.clip(np.asarray(probability, dtype=float), eps, 1.0 - eps)
    return np.log(p / (1.0 - p))


def sigmoid(values):
    values = np.clip(np.asarray(values, dtype=float), -40, 40)
    return 1.0 / (1.0 + np.exp(-values))


def logit_mean_pool(probability_matrix):
    """Mean log-odds pooling: average model evidence in logit space."""
    return sigmoid(probability_to_logit(np.asarray(probability_matrix)).mean(axis=0))


def correct_examples(accuracy, n_examples):
    return int(round(float(accuracy) * int(n_examples)))


def make_prediction_frame(source_df, final_prob, threshold, component_probabilities=None):
    """Rich per-row frame for Kaggle-side artifacts ONLY (contains raw text).

    Committable probability artifacts are produced by export_probability_artifact.
    """
    out = source_df[[id_col, SOURCE_TEXT_COL, MODEL_TEXT_COL, label_col, split_col]].copy()
    out = out.rename(
        columns={
            id_col: "row_id",
            SOURCE_TEXT_COL: "text",
            MODEL_TEXT_COL: "model_input_text",
            label_col: "y_true",
        }
    )
    out["y_prob"] = np.asarray(final_prob, dtype=float)
    out["threshold"] = float(threshold)
    out["y_pred"] = (out["y_prob"] >= threshold).astype(int)
    out["correct"] = out["y_true"] == out["y_pred"]
    out["error_type"] = "correct"
    out.loc[(out["y_true"] == 0) & (out["y_pred"] == 1), "error_type"] = "false_positive"
    out.loc[(out["y_true"] == 1) & (out["y_pred"] == 0), "error_type"] = "false_negative"
    out["text_hash"] = out["text"].map(sha256_text)
    if component_probabilities:
        for name, values in component_probabilities.items():
            out[name] = np.asarray(values, dtype=float)

    legacy_prediction_columns = [
        "row_id", "text", "y_true", "split", "y_prob", "threshold",
        "y_pred", "correct", "error_type", "text_hash",
    ]
    extra_prediction_columns = [
        column for column in out.columns if column not in legacy_prediction_columns
    ]
    return out[legacy_prediction_columns + extra_prediction_columns]


PROBS_FORBIDDEN_COLUMNS = ("text", "text_hash", "Original_Message", "text_preview", "message", "content", "raw_text")


def export_probability_artifact(source_df, y_prob, split_name, probs_dir):
    """Write the committable, text-free probability CSV for one split.

    Contract (tools/probs_artifact.py): exactly row_id, split, y_true, y_prob;
    canonical row ids; one split per file; probabilities finite in [0, 1].
    """
    artifact = pd.DataFrame({
        "row_id": source_df[id_col].astype(str).values,
        "split": split_name,
        "y_true": source_df[label_col].astype(int).values,
        "y_prob": np.asarray(y_prob, dtype=float),
    })
    if artifact["row_id"].duplicated().any():
        raise RuntimeError(f"Duplicate row_id in {split_name} probability artifact.")
    if not np.isfinite(artifact["y_prob"].to_numpy()).all():
        raise RuntimeError(f"Non-finite probabilities in {split_name} probability artifact.")
    if ((artifact["y_prob"] < 0) | (artifact["y_prob"] > 1)).any():
        raise RuntimeError(f"Probabilities outside [0, 1] in {split_name} probability artifact.")
    forbidden_present = [c for c in artifact.columns if c in PROBS_FORBIDDEN_COLUMNS]
    if forbidden_present:
        raise RuntimeError(f"Text-bearing columns in probability artifact: {forbidden_present}")
    path = Path(probs_dir) / f"{TECHNIQUE_NAME}__{split_name}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    artifact.to_csv(path, index=False)
    print(f"Wrote sanitized probability artifact: {path} ({len(artifact)} rows)")
    return path


In [ ]:
# ============================================================
# 8. Transformer dataset and deterministic dataloaders
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = [str(value) for value in texts]
        self.labels = None if labels is None else np.asarray(labels).astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        item = {"text": self.texts[index]}
        if self.labels is not None:
            item["label"] = int(self.labels[index])
        return item


def make_collate_fn(tokenizer, max_length):
    def collate_fn(batch):
        encoded = tokenizer(
            [item["text"] for item in batch],
            truncation=True,
            padding=True,
            max_length=int(max_length),
            return_tensors="pt",
        )
        if "label" in batch[0]:
            encoded["labels"] = torch.tensor([item["label"] for item in batch], dtype=torch.long)
        return encoded
    return collate_fn


def make_dataloader(df_part, tokenizer, batch_size, max_length, shuffle=False, seed=30):
    dataset = TextClassificationDataset(
        texts=df_part[MODEL_TEXT_COL].tolist(),
        labels=df_part[label_col].tolist(),
    )
    generator = torch.Generator()
    generator.manual_seed(int(seed))
    return DataLoader(
        dataset,
        batch_size=int(batch_size),
        shuffle=bool(shuffle),
        collate_fn=make_collate_fn(tokenizer, max_length),
        num_workers=int(CONFIG["transformer_training"]["num_workers"]),
        drop_last=bool(CONFIG["transformer_training"]["drop_last"]),
        pin_memory=torch.cuda.is_available(),
        worker_init_fn=seed_worker,
        generator=generator,
    )


In [ ]:
# ============================================================
# 9. Model loading with checkpoint-specific classifier-head policy
# ============================================================

def reinitialize_classification_head(model):
    head_name = None
    head = None
    for candidate in ["classifier", "score", "classification_head"]:
        if hasattr(model, candidate):
            head_name = candidate
            head = getattr(model, candidate)
            break
    if head is None:
        raise AttributeError(
            "Could not identify the classification head. Add this architecture's head attribute to the helper."
        )

    if hasattr(model, "_init_weights"):
        head.apply(model._init_weights)
    else:
        for module in head.modules():
            if hasattr(module, "reset_parameters"):
                module.reset_parameters()
    return head_name


def floating_parameter_dtypes(model):
    counts = {}
    for parameter in model.parameters():
        if parameter.requires_grad and parameter.is_floating_point():
            name = str(parameter.dtype).replace("torch.", "")
            counts[name] = counts.get(name, 0) + 1
    return counts


def checkpoint_head_policy(model_name_or_path):
    exact = CONFIG.get("checkpoint_head_policy", {}).get(model_name_or_path)
    policy = exact or CONFIG.get("default_head_policy", "fresh")
    policy = str(policy).strip().lower()
    if policy not in {"pretrained", "fresh"}:
        raise ValueError(
            f"Head policy for {model_name_or_path!r} must be 'pretrained' or 'fresh', got {policy!r}."
        )
    return policy


def load_tokenizer_and_model(model_name_or_path, random_seed):
    set_global_seed(random_seed)
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name_or_path,
        num_labels=2,
        id2label={0: "non_extremist", 1: "extremist"},
        label2id={"non_extremist": 0, "extremist": 1},
        ignore_mismatched_sizes=True,
    )

    if CONFIG["transformer_training"].get("force_fp32_master_weights", True):
        model = model.float()

    head_policy = checkpoint_head_policy(model_name_or_path)
    pretrained_head_labels = CONFIG.get("checkpoint_pretrained_head_labels", {}).get(
        model_name_or_path
    )
    if head_policy == "pretrained" and pretrained_head_labels != 2:
        # A pretrained head can be kept only when the source checkpoint's head is
        # binary with the extremist-aligned class at index 1; otherwise
        # ignore_mismatched_sizes has already replaced it with a fresh head.
        raise ValueError(
            f"Head policy 'pretrained' requires a declared 2-label source head for "
            f"{model_name_or_path!r}; declare checkpoint_pretrained_head_labels or use 'fresh'."
        )

    reset_head_name = None
    if head_policy == "fresh":
        reset_head_name = reinitialize_classification_head(model)

    dtype_counts = floating_parameter_dtypes(model)
    non_fp32 = {dtype: count for dtype, count in dtype_counts.items() if dtype != "float32"}
    if non_fp32:
        raise RuntimeError(
            "Trainable transformer parameters must be FP32 before optimizer creation. "
            f"Observed non-FP32 parameter dtypes: {non_fp32}"
        )

    print(
        f"Loaded {model_name_or_path} with classifier-head policy={head_policy}; "
        f"reinitialized_head={reset_head_name}"
    )
    return tokenizer, model, reset_head_name, head_policy


def swap_binary_head_rows(model):
    """Swap a binary head's output rows (label-orientation fix from the probe)."""
    head_linear = None
    classifier = getattr(model, "classifier", None)
    if classifier is not None:
        head_linear = getattr(classifier, "out_proj", None)
        if head_linear is None and isinstance(classifier, nn.Linear):
            head_linear = classifier
    if head_linear is None or head_linear.weight.shape[0] != 2:
        raise AttributeError("Could not locate a binary head linear layer to swap.")
    with torch.no_grad():
        head_linear.weight.data = head_linear.weight.data[[1, 0], :].clone()
        if head_linear.bias is not None:
            head_linear.bias.data = head_linear.bias.data[[1, 0]].clone()
    return True


def resolve_encoder_layers(model):
    """Locate the transformer's encoder layer stack for layer-wise LR decay."""
    for backbone_attr in ["roberta", "deberta", "bert", "base_model"]:
        backbone = getattr(model, backbone_attr, None)
        if backbone is not None and hasattr(backbone, "encoder"):
            encoder = backbone.encoder
            if hasattr(encoder, "layer"):
                return list(encoder.layer)
    raise AttributeError("Could not locate encoder layers for layer-wise LR decay.")


def build_optimizer(model, learning_rate, weight_decay, layerwise_lr_decay=None):
    """AdamW, optionally with layer-wise learning-rate decay.

    With decay d and N encoder layers, layer i (0 = bottom) trains at
    lr * d^(N - i); embeddings train at lr * d^(N + 1); head/pooler at lr.
    """
    if not layerwise_lr_decay:
        return torch.optim.AdamW(
            model.parameters(), lr=float(learning_rate), weight_decay=float(weight_decay)
        )

    decay = float(layerwise_lr_decay)
    layers = resolve_encoder_layers(model)
    num_layers = len(layers)
    layer_ids = {id(p): i for i, layer in enumerate(layers) for p in layer.parameters()}
    embedding_param_ids = set()
    for backbone_attr in ["roberta", "deberta", "bert", "base_model"]:
        backbone = getattr(model, backbone_attr, None)
        if backbone is not None and hasattr(backbone, "embeddings"):
            embedding_param_ids = {id(p) for p in backbone.embeddings.parameters()}
            break

    groups = {}
    for parameter in model.parameters():
        if not parameter.requires_grad:
            continue
        if id(parameter) in embedding_param_ids:
            scale = decay ** (num_layers + 1)
        elif id(parameter) in layer_ids:
            scale = decay ** (num_layers - layer_ids[id(parameter)])
        else:
            scale = 1.0
        groups.setdefault(scale, []).append(parameter)

    param_groups = [
        {"params": params, "lr": float(learning_rate) * scale}
        for scale, params in sorted(groups.items())
    ]
    return torch.optim.AdamW(param_groups, lr=float(learning_rate), weight_decay=float(weight_decay))


In [ ]:
# ============================================================
# 10. Transformer training and inference (FGM, EMA, accuracy@0.50 monitor)
# ============================================================

def resolve_amp_runtime(amp_dtype_preference=None):
    """Resolve a safe AMP policy for the current accelerator.

    Explicit per-checkpoint preference beats the global default. bfloat16 on a
    T4 is software-emulated (no tensor cores) but numerically safer for
    fp16-fragile architectures; float16 + GradScaler is the fast default.
    """
    requested = bool(CONFIG["transformer_training"].get("use_amp_if_cuda", True))
    if not requested or DEVICE.type != "cuda":
        return {
            "enabled": False,
            "dtype": torch.float32,
            "dtype_name": "float32",
            "use_grad_scaler": False,
        }

    preference = str(
        amp_dtype_preference
        or CONFIG["transformer_training"].get("amp_dtype", "float16")
    ).strip().lower()
    if preference not in {"float32", "bfloat16", "float16"}:
        raise ValueError("amp_dtype must be one of: float32, bfloat16, float16")

    if preference == "float32":
        return {
            "enabled": False,
            "dtype": torch.float32,
            "dtype_name": "float32",
            "use_grad_scaler": False,
        }

    bf16_supported = bool(
        hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported()
    )
    if preference == "bfloat16" and not bf16_supported:
        warnings.warn(
            "BF16 AMP was requested but is not supported by this GPU; falling back to FP16 AMP."
        )
        preference = "float16"

    if preference == "bfloat16":
        amp_dtype = torch.bfloat16
        use_grad_scaler = False  # BF16 keeps FP32 exponent range; no loss scaling needed.
    else:
        amp_dtype = torch.float16
        use_grad_scaler = True

    return {
        "enabled": True,
        "dtype": amp_dtype,
        "dtype_name": str(amp_dtype).replace("torch.", ""),
        "use_grad_scaler": use_grad_scaler,
    }


def amp_autocast_context(amp_runtime):
    if not amp_runtime["enabled"]:
        return nullcontext()
    return torch.autocast(device_type=DEVICE.type, dtype=amp_runtime["dtype"], enabled=True)


def make_grad_scaler(enabled):
    """Support both current torch.amp and older torch.cuda.amp APIs."""
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def verify_fp32_optimizer_parameters(model):
    bad = []
    for name, parameter in model.named_parameters():
        if (
            parameter.requires_grad
            and parameter.is_floating_point()
            and parameter.dtype != torch.float32
        ):
            bad.append((name, str(parameter.dtype)))
    if bad:
        preview = bad[:10]
        raise RuntimeError(
            "Optimizer parameters must remain FP32 when using AMP. "
            f"Found {len(bad)} non-FP32 trainable parameters; first entries: {preview}"
        )


class FGMPerturbation:
    """Fast Gradient Method on the word-embedding matrix.

    After the clean backward pass, perturb the embedding weights by
    eps * g / ||g||_2 (direction is invariant to GradScaler's loss-scale factor),
    accumulate the adversarial loss's gradients, then restore the weights.
    """

    def __init__(self, model, epsilon):
        self.epsilon = float(epsilon)
        self.embedding = model.get_input_embeddings()
        self.backup = None

    def attack(self):
        weight = self.embedding.weight
        if weight.grad is None:
            return False
        grad = weight.grad.detach()
        if not torch.isfinite(grad).all():
            return False
        norm = torch.norm(grad)
        if norm == 0 or not torch.isfinite(norm):
            return False
        self.backup = weight.data.clone()
        weight.data.add_(self.epsilon * grad / norm)
        return True

    def restore(self):
        if self.backup is not None:
            self.embedding.weight.data.copy_(self.backup)
            self.backup = None


class ExponentialMovingAverage:
    """EMA shadow of all floating-point parameters, applied for evaluation."""

    def __init__(self, model, decay):
        self.decay = float(decay)
        self.shadow = {
            name: parameter.detach().clone()
            for name, parameter in model.named_parameters()
            if parameter.requires_grad and parameter.is_floating_point()
        }
        self.backup = {}

    def update(self, model):
        with torch.no_grad():
            for name, parameter in model.named_parameters():
                if name in self.shadow:
                    self.shadow[name].mul_(self.decay).add_(
                        parameter.detach(), alpha=1.0 - self.decay
                    )

    def apply_to(self, model):
        self.backup = {}
        with torch.no_grad():
            for name, parameter in model.named_parameters():
                if name in self.shadow:
                    self.backup[name] = parameter.detach().clone()
                    parameter.data.copy_(self.shadow[name])

    def restore(self, model):
        with torch.no_grad():
            for name, parameter in model.named_parameters():
                if name in self.backup:
                    parameter.data.copy_(self.backup[name])
        self.backup = {}


def resolve_class_weight_tensor(class_weight_spec, labels, num_labels=2):
    """Resolve a class-weight specification into a FP32 tensor on DEVICE."""
    if class_weight_spec is None:
        return None, None

    y = np.asarray(labels, dtype=int)
    if y.ndim != 1 or len(y) == 0:
        raise ValueError("Training labels must be a non-empty one-dimensional array.")
    if y.min() < 0 or y.max() >= int(num_labels):
        raise ValueError(
            f"Training labels must be integers in [0, {int(num_labels) - 1}]. "
            f"Observed range: [{int(y.min())}, {int(y.max())}]"
        )

    if isinstance(class_weight_spec, str):
        if class_weight_spec.strip().lower() != "balanced":
            raise ValueError(
                "class_weight must be None, 'balanced', a dict, or a sequence of weights. "
                f"Received: {class_weight_spec!r}"
            )
        counts = np.bincount(y, minlength=int(num_labels)).astype(np.float64)
        if np.any(counts == 0):
            missing = np.flatnonzero(counts == 0).tolist()
            raise ValueError(
                f"Cannot compute balanced class weights because training classes {missing} have zero examples."
            )
        weights = len(y) / (float(num_labels) * counts)
    elif isinstance(class_weight_spec, dict):
        weights = np.asarray(
            [class_weight_spec.get(i, class_weight_spec.get(str(i), np.nan)) for i in range(int(num_labels))],
            dtype=np.float64,
        )
    else:
        weights = np.asarray(class_weight_spec, dtype=np.float64)

    if weights.shape != (int(num_labels),):
        raise ValueError(f"Expected exactly {int(num_labels)} class weights; received shape {weights.shape}.")
    if not np.all(np.isfinite(weights)) or np.any(weights <= 0):
        raise ValueError(f"Class weights must be finite and strictly positive. Received: {weights.tolist()}")

    weight_tensor = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
    return weight_tensor, [float(value) for value in weights]


def predict_transformer_proba(model, dataloader, amp_dtype_preference=None):
    model.eval()
    probabilities, labels = [], []
    amp_runtime = resolve_amp_runtime(amp_dtype_preference)
    with torch.inference_mode():
        for batch in dataloader:
            batch = {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}
            y = batch.pop("labels", None)
            with amp_autocast_context(amp_runtime):
                outputs = model(**batch)
                batch_prob = torch.softmax(outputs.logits.float(), dim=-1)[:, 1]
            probabilities.extend(batch_prob.detach().cpu().numpy().tolist())
            if y is not None:
                labels.extend(y.detach().cpu().numpy().tolist())
    return np.asarray(probabilities, dtype=float), np.asarray(labels, dtype=int)


class NonFiniteTrainingError(RuntimeError):
    """Raised when a run hits non-finite loss/grad; callers may degrade, not die."""


def train_transformer_config(config_params, run_label="config", random_seed=None):
    """Train one (checkpoint, hyperparameters, regularizers) configuration.

    config_params keys: checkpoint, learning_rate, batch_size, max_length,
    weight_decay, label_smoothing, class_weight, num_epochs, plus optional
    gradient_accumulation_steps, layerwise_lr_decay, amp_dtype, use_fgm,
    fgm_epsilon, use_ema, ema_decay, gradient_checkpointing.
    """
    seed = CONFIG["random_seed"] if random_seed is None else int(random_seed)
    checkpoint = config_params["checkpoint"]

    # A previous failed CUDA run can retain references through a notebook traceback.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    tokenizer, model, reset_head_name, head_policy = load_tokenizer_and_model(checkpoint, seed)
    if config_params.get("orientation_swap", False):
        swap_binary_head_rows(model)
        print(f"{run_label} | applied head-orientation swap (probe decision)")
    model.to(device=DEVICE, dtype=torch.float32)
    if config_params.get("gradient_checkpointing", False):
        model.gradient_checkpointing_enable()
    verify_fp32_optimizer_parameters(model)

    train_loader = make_dataloader(
        train_df, tokenizer,
        batch_size=config_params["batch_size"],
        max_length=config_params["max_length"],
        shuffle=True,
        seed=seed,
    )
    val_loader = make_dataloader(
        val_df, tokenizer,
        batch_size=config_params["batch_size"],
        max_length=config_params["max_length"],
        shuffle=False,
        seed=seed,
    )

    class_weight_tensor, resolved_class_weights = resolve_class_weight_tensor(
        config_params.get("class_weight"), train_df[label_col].to_numpy(), num_labels=2
    )
    criterion = nn.CrossEntropyLoss(
        weight=class_weight_tensor,
        label_smoothing=float(config_params.get("label_smoothing", 0.0)),
    )
    optimizer = build_optimizer(
        model,
        learning_rate=config_params["learning_rate"],
        weight_decay=config_params["weight_decay"],
        layerwise_lr_decay=config_params.get("layerwise_lr_decay"),
    )

    grad_accum = int(
        config_params.get(
            "gradient_accumulation_steps",
            CONFIG["transformer_training"]["gradient_accumulation_steps"],
        )
    )
    if grad_accum < 1:
        raise ValueError("gradient_accumulation_steps must be at least 1")

    num_epochs = int(config_params["num_epochs"])
    total_update_steps = max(1, math.ceil(len(train_loader) / grad_accum) * num_epochs)
    warmup_steps = int(CONFIG["transformer_training"]["warmup_ratio"] * total_update_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_update_steps
    )

    amp_runtime = resolve_amp_runtime(config_params.get("amp_dtype"))
    scaler = make_grad_scaler(enabled=amp_runtime["use_grad_scaler"])

    use_fgm = bool(config_params.get("use_fgm", False))
    fgm = FGMPerturbation(model, config_params.get("fgm_epsilon", 1.0)) if use_fgm else None
    use_ema = bool(config_params.get("use_ema", False))
    ema = ExponentialMovingAverage(model, config_params.get("ema_decay", 0.995)) if use_ema else None

    print(
        f"{run_label} | device={DEVICE} | AMP={amp_runtime['enabled']} | "
        f"AMP dtype={amp_runtime['dtype_name']} | GradScaler={amp_runtime['use_grad_scaler']} | "
        f"FGM={use_fgm} | EMA={use_ema} | LLRD={config_params.get('layerwise_lr_decay')} | "
        f"class_weight={config_params.get('class_weight')} | resolved_weights={resolved_class_weights} | "
        f"trainable parameter dtypes={floating_parameter_dtypes(model)}"
    )

    best_monitor_key = None
    best_epoch = None
    best_state_dict = None
    best_val_prob = None
    best_val_metrics_at_050 = None
    training_history = []
    patience = int(CONFIG["transformer_training"]["early_stopping_patience_epochs"])
    epochs_without_improvement = 0

    train_batches = len(train_loader)
    final_window_size = train_batches % grad_accum

    nan_guard_limit = int(
        CONFIG["transformer_training"]["nan_guard_events_before_fp32_restart"]
    )

    for epoch in range(1, num_epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_unscaled_loss = 0.0
        nonfinite_events_this_epoch = 0

        for step, batch in enumerate(train_loader, start=1):
            batch = {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}
            labels = batch.pop("labels")

            # Correctly normalize a final partial accumulation window instead of
            # underweighting it when len(train_loader) is not divisible by grad_accum.
            in_partial_final_window = (
                final_window_size > 0 and step > train_batches - final_window_size
            )
            accumulation_divisor = final_window_size if in_partial_final_window else grad_accum

            with amp_autocast_context(amp_runtime):
                outputs = model(**batch)
                unscaled_loss = criterion(outputs.logits.float(), labels)
                loss = unscaled_loss / accumulation_divisor

            if not torch.isfinite(unscaled_loss):
                # A single fp16 overflow is survivable (skip the batch, let the
                # scaler rescale); repeated events mean the run is unstable and
                # the caller decides between an fp32 restart and dropping it.
                nonfinite_events_this_epoch += 1
                optimizer.zero_grad(set_to_none=True)
                if nonfinite_events_this_epoch >= nan_guard_limit:
                    raise NonFiniteTrainingError(
                        f"{nonfinite_events_this_epoch} non-finite losses in one epoch "
                        f"({run_label}, epoch={epoch}, step={step})."
                    )
                continue

            if amp_runtime["use_grad_scaler"]:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            if fgm is not None and fgm.attack():
                with amp_autocast_context(amp_runtime):
                    adv_outputs = model(**batch)
                    adv_loss = criterion(adv_outputs.logits.float(), labels) / accumulation_divisor
                if amp_runtime["use_grad_scaler"]:
                    scaler.scale(adv_loss).backward()
                else:
                    adv_loss.backward()
                fgm.restore()

            total_unscaled_loss += float(unscaled_loss.detach().cpu())

            should_update = step % grad_accum == 0 or step == train_batches
            if should_update:
                if amp_runtime["use_grad_scaler"]:
                    scaler.unscale_(optimizer)

                grad_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), float(CONFIG["transformer_training"]["max_grad_norm"])
                )
                if not torch.isfinite(grad_norm) and not amp_runtime["use_grad_scaler"]:
                    optimizer.zero_grad(set_to_none=True)
                    raise NonFiniteTrainingError(
                        f"Non-finite gradient norm in {run_label}, epoch={epoch}, step={step}: "
                        f"{float(grad_norm.detach().cpu())}"
                    )

                if amp_runtime["use_grad_scaler"]:
                    # GradScaler.step skips the update on inf/NaN grads and rescales.
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()

                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                if ema is not None:
                    ema.update(model)

        # Evaluate the weights that would actually ship: EMA shadow in EMA runs.
        if ema is not None:
            ema.apply_to(model)
        val_prob, val_true = predict_transformer_proba(
            model, val_loader, config_params.get("amp_dtype")
        )
        state_for_epoch = {
            key: value.detach().cpu().clone() for key, value in model.state_dict().items()
        }
        if ema is not None:
            ema.restore(model)

        val_metrics_050 = compute_binary_metrics(val_true, val_prob, 0.50, CONFIG["positive_label"])

        # Epoch monitor = accuracy at the fixed 0.50 operating point: the monitor
        # measures exactly what the deployed decision rule will use (nb10 monitored
        # positive_f1 while documenting accuracy — that contradiction is removed).
        monitor_key = (
            float(val_metrics_050["accuracy"]),
            float(val_metrics_050["balanced_accuracy"]),
            float(val_metrics_050["positive_f1"]),
            float(val_metrics_050["pr_auc"]),
            -float(val_metrics_050["brier_score"]),
        )

        history_row = {
            "run_label": run_label,
            "random_seed": seed,
            "checkpoint": checkpoint,
            "classifier_head_policy": head_policy,
            "classifier_head_reinitialized": reset_head_name,
            "amp_enabled": amp_runtime["enabled"],
            "amp_dtype": amp_runtime["dtype_name"],
            "grad_scaler_enabled": amp_runtime["use_grad_scaler"],
            "use_fgm": use_fgm,
            "use_ema": use_ema,
            "layerwise_lr_decay": config_params.get("layerwise_lr_decay"),
            "class_weight_spec": config_params.get("class_weight"),
            "epoch": epoch,
            "train_loss_mean": total_unscaled_loss / max(1, len(train_loader)),
            "learning_rate": optimizer.param_groups[0]["lr"],
            **{f"val_at_050_{key}": value for key, value in val_metrics_050.items()},
        }
        training_history.append(history_row)

        print(
            f"{run_label} | seed={seed} | epoch {epoch}/{num_epochs} | "
            f"loss={history_row['train_loss_mean']:.4f} | "
            f"val_acc@0.50={val_metrics_050['accuracy']:.4f} | "
            f"val_pr_auc={val_metrics_050['pr_auc']:.4f}"
        )

        improvement = best_monitor_key is None or monitor_key > best_monitor_key
        if improvement:
            best_monitor_key = monitor_key
            best_epoch = epoch
            best_state_dict = state_for_epoch
            best_val_prob = val_prob.copy()
            best_val_metrics_at_050 = val_metrics_050.copy()
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping after epoch {epoch}; best epoch was {best_epoch}.")
                break

    if best_state_dict is None:
        raise RuntimeError("Training finished without a valid model state.")
    model.load_state_dict(best_state_dict)
    model.to(device=DEVICE, dtype=torch.float32)
    verify_fp32_optimizer_parameters(model)

    return {
        "tokenizer": tokenizer,
        "model": model,
        "best_epoch": best_epoch,
        "best_monitor_metric": float(best_monitor_key[0]),
        "best_val_prob": best_val_prob,
        "best_val_metrics_at_050": best_val_metrics_at_050,
        "training_history": pd.DataFrame(training_history),
        "random_seed": seed,
        "checkpoint": checkpoint,
        "reset_head_name": reset_head_name,
        "head_policy": head_policy,
        "amp_runtime": amp_runtime,
    }


In [ ]:
# ============================================================
# 11. Baseline reference, component resolution, and head-orientation probe
# ============================================================

y_train = train_df[label_col].to_numpy(dtype=int)
y_val = val_df[label_col].to_numpy(dtype=int)
y_test = test_df[label_col].to_numpy(dtype=int)

majority_class = int(pd.Series(y_train).mode().iloc[0])
baseline_metrics = {
    "validation_majority_class_accuracy": float(
        accuracy_score(y_val, np.full(len(y_val), majority_class))
    ),
    "test_majority_class_accuracy": float(
        accuracy_score(y_test, np.full(len(y_test), majority_class))
    ),
    "majority_class": majority_class,
    "train_positive_rate": float((y_train == CONFIG["positive_label"]).mean()),
    "validation_positive_rate": float((y_val == CONFIG["positive_label"]).mean()),
    "test_positive_rate": float((y_test == CONFIG["positive_label"]).mean()),
}
save_json(baseline_metrics, OUTPUT_DIR / "baseline_reference_metrics.json")
print(json.dumps(baseline_metrics, indent=2))

# ------------------------------------------------------------------
# Resolve every component's checkpoint availability up front (downloads cache
# to disk), applying the prespecified download-failure policy. No checkpoint is
# ever substituted mid-run for a *quality* failure — only for a download failure.
# ------------------------------------------------------------------
component_status = {}       # name -> {"checkpoint", "head_policy", "active", "note"}
component_drop_log = []


def try_download_checkpoint(checkpoint):
    tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint, num_labels=2, ignore_mismatched_sizes=True
    )
    del tokenizer, model
    gc.collect()
    return True


for component_name, component in CONFIG["components"].items():
    checkpoint = component["checkpoint"]
    head_policy = component["head_policy"]
    active, note = True, "primary checkpoint available"
    try:
        try_download_checkpoint(checkpoint)
    except Exception as primary_error:
        policy = component["on_download_failure"]
        if policy == "hard_abort":
            raise RuntimeError(
                f"Anchor checkpoint {checkpoint!r} is unavailable: {primary_error}. "
                "The design is meaningless without its floor; aborting before any training."
            )
        fallback = component.get("download_fallback")
        if policy == "try_fallback_then_drop" and fallback is not None:
            try:
                try_download_checkpoint(fallback)
                checkpoint = fallback
                head_policy = component.get("fallback_head_policy", "fresh")
                note = f"primary unavailable ({primary_error}); using fallback {fallback}"
            except Exception as fallback_error:
                active = False
                note = f"primary and fallback unavailable ({primary_error}; {fallback_error})"
        else:
            active = False
            note = f"unavailable ({primary_error})"
    if not active:
        component_drop_log.append({
            "component": component_name, "stage": "download", "reason": note,
        })
    component_status[component_name] = {
        "checkpoint": checkpoint,
        "head_policy": head_policy,
        "active": active,
        "note": note,
    }
    print(f"{component_name}: active={active} | checkpoint={checkpoint} | {note}")

# ------------------------------------------------------------------
# Head-orientation probe (train split only): a kept pretrained binary head must
# emit our positive class (extremist-aligned) on logits column 1. Zero-shot AUC
# below 0.5 on train rows means the head is inverted; record the fix so every
# later load of that component swaps the head's output rows.
# ------------------------------------------------------------------
orientation_swap_needed = {}

for component_name, component in CONFIG["components"].items():
    status = component_status[component_name]
    if not (status["active"] and component.get("orientation_probe", False)):
        continue
    if status["head_policy"] != "pretrained":
        continue
    probe_rows = int(component.get("orientation_probe_rows", 512))
    probe_df = train_df.head(probe_rows)
    tokenizer = AutoTokenizer.from_pretrained(status["checkpoint"], use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        status["checkpoint"], num_labels=2, ignore_mismatched_sizes=True
    ).to(DEVICE).float()
    probe_loader = make_dataloader(
        probe_df, tokenizer,
        batch_size=component["batch_size"],
        max_length=CONFIG["max_length"],
        shuffle=False,
        seed=CONFIG["random_seed"],
    )
    probe_prob, probe_true = predict_transformer_proba(model, probe_loader)
    probe_auc = safe_roc_auc(probe_true, probe_prob)
    orientation_swap_needed[component_name] = bool(probe_auc < 0.5)
    print(
        f"Orientation probe {component_name}: zero-shot train AUC={probe_auc:.4f} "
        f"-> swap_head={orientation_swap_needed[component_name]}"
    )
    del model, tokenizer, probe_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

save_json(
    {
        "component_status": component_status,
        "orientation_swap_needed": orientation_swap_needed,
        "drop_log": component_drop_log,
    },
    OUTPUT_DIR / "auxiliary" / "component_resolution.json",
)


In [ ]:
# ============================================================
# 12. Validation-only learning-rate screening (fixed 0.50 operating point)
# ============================================================

def build_run_config(component_name, learning_rate, amp_dtype=None,
                     gradient_checkpointing=None, batch_size=None, grad_accum=None):
    """Assemble one training-run configuration from a component + shared recipe."""
    component = CONFIG["components"][component_name]
    status = component_status[component_name]
    shared = CONFIG["transformer_training"]
    precision = component["precision"]
    if amp_dtype is None:
        amp_dtype = "float32" if precision == "fp32" else "float16"
    return {
        "component": component_name,
        "checkpoint": status["checkpoint"],
        "learning_rate": float(learning_rate),
        "batch_size": int(batch_size if batch_size is not None else component["batch_size"]),
        "max_length": int(CONFIG["max_length"]),
        "weight_decay": float(shared["weight_decay"]),
        "label_smoothing": float(shared["label_smoothing"]),
        "class_weight": shared["class_weight"],
        "num_epochs": int(component["num_epochs"]),
        "gradient_accumulation_steps": int(
            grad_accum if grad_accum is not None else component["gradient_accumulation_steps"]
        ),
        "gradient_checkpointing": bool(
            gradient_checkpointing if gradient_checkpointing is not None
            else component["gradient_checkpointing"]
        ),
        "amp_dtype": amp_dtype,
        "use_fgm": bool(component.get("fgm", False)),
        "fgm_epsilon": float(CONFIG["fgm"]["epsilon"]),
        "use_ema": False,
        "orientation_swap": bool(orientation_swap_needed.get(component_name, False)),
    }


def run_with_degradation(component_name, run_config, run_label, seed):
    """Train one run with the prespecified failure ladder.

    OOM -> component's oom_ladder steps (checkpointing on, then smaller batch);
    repeated non-finite losses -> one fp32 restart; anything after that -> the
    run is dropped and admission judges the component on its remaining seeds.
    Returns the training-result dict, or None if the run is dropped.
    """
    component = CONFIG["components"][component_name]
    attempts = [run_config]
    for oom_step in component.get("oom_ladder", []):
        previous = attempts[-1]
        if oom_step == "gradient_checkpointing_on":
            attempts.append({**previous, "gradient_checkpointing": True})
        elif oom_step == "batch_size_4_accum_4":
            attempts.append({**previous, "gradient_checkpointing": True,
                             "batch_size": 4, "gradient_accumulation_steps": 4})

    attempt_index = 0
    fp32_restarts_used = 0
    current = attempts[0]
    while True:
        try:
            result = train_transformer_config(current, run_label=run_label, random_seed=seed)
            # The attempt that actually finished (post-ladder, post-restart) is
            # what test inference must reproduce — not the original request.
            result["effective_run_config"] = current
            return result
        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
            attempt_index += 1
            if attempt_index < len(attempts):
                current = {**attempts[attempt_index], "amp_dtype": current["amp_dtype"]}
                print(f"{run_label} | OOM -> ladder step {attempt_index}: "
                      f"bs={current['batch_size']} ckpt={current['gradient_checkpointing']}")
                continue
            print(f"{run_label} | OOM after full ladder -> run dropped")
            component_drop_log.append({
                "component": component_name, "stage": run_label, "reason": "oom_after_ladder",
            })
            return None
        except NonFiniteTrainingError as error:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            max_restarts = int(CONFIG["transformer_training"]["fp32_restart_max_attempts"])
            if current["amp_dtype"] != "float32" and fp32_restarts_used < max_restarts:
                fp32_restarts_used += 1
                current = {**current, "amp_dtype": "float32"}
                print(f"{run_label} | {error} -> restarting once in fp32")
                continue
            print(f"{run_label} | {error} -> run dropped")
            component_drop_log.append({
                "component": component_name, "stage": run_label, "reason": f"non_finite: {error}",
            })
            return None


def compute_test_probs(result, run_config):
    """Inference-only test probabilities for a finished run. No metric is computed.

    Uses the run's EFFECTIVE configuration and resolved AMP policy so that test
    inference exactly matches the precision that produced the run's validation
    probabilities (an fp32-restarted run must not infer test in fp16).
    """
    effective = result.get("effective_run_config", run_config)
    test_loader = make_dataloader(
        test_df, result["tokenizer"],
        batch_size=effective["batch_size"],
        max_length=effective["max_length"],
        shuffle=False,
        seed=result["random_seed"],
    )
    test_prob_run, test_true_run = predict_transformer_proba(
        result["model"], test_loader, result["amp_runtime"]["dtype_name"]
    )
    if not np.array_equal(test_true_run, y_test):
        raise RuntimeError("Test-label order changed during transformer inference.")
    del test_loader
    return test_prob_run


screen_store = {}        # (component_name, learning_rate) -> stored run payload
screen_rows = []
screen_histories = []
screen_start = time.time()
SCREEN_SEED = int(CONFIG["screening"]["screen_seed"])

for component_name, component in CONFIG["components"].items():
    if not component_status[component_name]["active"]:
        continue
    lr_candidates = component.get("lr_candidates")
    if not lr_candidates:
        continue

    for lr in lr_candidates:
        run_label = f"screen_{component_name}_lr{lr:.0e}"
        print("\n" + "=" * 90)
        print(f"Screening {run_label}")
        config_start = time.time()
        run_config = build_run_config(component_name, lr)
        result = run_with_degradation(component_name, run_config, run_label, SCREEN_SEED)

        if result is None:
            screen_rows.append({
                "component": component_name, "checkpoint": run_config["checkpoint"],
                "learning_rate": lr, "status": "failed",
                "runtime_seconds": time.time() - config_start,
            })
            continue

        # Screening is ranked at the deployed operating point (fixed 0.50);
        # no per-config threshold tuning anywhere.
        val_metrics = result["best_val_metrics_at_050"]
        test_prob_run = compute_test_probs(result, run_config)
        screen_store[(component_name, float(lr))] = {
            "val_prob": result["best_val_prob"].copy(),
            "test_prob": test_prob_run,
            "best_epoch": result["best_epoch"],
            "history": result["training_history"].copy(),
            "run_config": run_config,
        }

        history = result["training_history"].copy()
        history["member"] = component_name
        history["screen_learning_rate"] = lr
        screen_histories.append(history)

        screen_rows.append({
            "component": component_name, "checkpoint": run_config["checkpoint"],
            "learning_rate": lr, "status": "ok",
            "best_epoch": result["best_epoch"],
            "runtime_seconds": time.time() - config_start,
            **val_metrics,
        })

        del result
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

screening_results = pd.DataFrame(screen_rows)
if not screening_results.empty and "accuracy" in screening_results.columns:
    screening_results = screening_results.sort_values(
        ["component", "accuracy", "balanced_accuracy", "positive_f1", "pr_auc"],
        ascending=[True, False, False, False, False],
    ).reset_index(drop=True)
screening_results.to_csv(OUTPUT_DIR / "ablation" / "ablation_results.csv", index=False)

if screen_histories:
    pd.concat(screen_histories, ignore_index=True).to_csv(
        OUTPUT_DIR / "training_logs" / "ablation_training_history.csv", index=False
    )

print("\nScreening runtime minutes:", round((time.time() - screen_start) / 60, 2))
display(screening_results)


In [ ]:
# ============================================================
# 13. Select each screened component's learning rate
# ============================================================

final_component_configs = {}

for component_name, component in CONFIG["components"].items():
    status = component_status[component_name]
    if not status["active"]:
        continue

    if not component.get("lr_candidates"):
        # A and F carry nb10's screened winner unchanged; re-screening a
        # thrice-swept grid is pure validation-noise spend.
        final_component_configs[component_name] = build_run_config(
            component_name, component["learning_rate"]
        )
        continue

    candidate_rows = [
        row for row in screen_rows
        if row["component"] == component_name and row.get("status") == "ok"
    ]
    if not candidate_rows:
        status["active"] = False
        component_drop_log.append({
            "component": component_name, "stage": "screening",
            "reason": "no learning-rate candidate finished",
        })
        print(f"{component_name}: dropped (no successful screening run)")
        continue

    winner = max(
        candidate_rows,
        key=lambda row: (
            row["accuracy"], row["balanced_accuracy"], row["positive_f1"], row["pr_auc"],
        ),
    )
    final_component_configs[component_name] = build_run_config(
        component_name, winner["learning_rate"]
    )
    print(
        f"{component_name}: selected lr={winner['learning_rate']:.0e} "
        f"(val acc@0.50={winner['accuracy']:.4f}, best_epoch={winner['best_epoch']})"
    )

print("\nFinal component configurations:")
for component_name, run_config in final_component_configs.items():
    print(
        f"  {component_name}: {run_config['checkpoint']} | lr={run_config['learning_rate']:.0e} | "
        f"bs={run_config['batch_size']}x{run_config['gradient_accumulation_steps']} | "
        f"fgm={run_config['use_fgm']} | amp={run_config['amp_dtype']}"
    )


In [ ]:
# ============================================================
# 14. Final training queue with per-run val/test probabilities
# ============================================================

run_records = []            # one dict per finished run
final_histories = []
representative_model = None
representative_tokenizer = None
representative_seed = None

REPRESENTATIVE_COMPONENT = CONFIG["explainability"]["representative_component"]
PREFERRED_REPRESENTATIVE_SEED = int(CONFIG["explainability"]["representative_seed"])
TIME_BUDGET = CONFIG["time_budget"]

FINAL_QUEUE = [
    "A_anchor", "B_dynabench", "C_hatebert", "D_deberta", "E_twitter_large", "F_anchor_fgm",
]

for component_name in FINAL_QUEUE:
    if component_name not in final_component_configs:
        continue
    if not component_status[component_name]["active"]:
        continue
    component = CONFIG["components"][component_name]
    run_config = final_component_configs[component_name]

    # Prespecified time-budget degradation: F is the first casualty, then E.
    elapsed_seconds = time.time() - RUN_START_TIME
    if component_name == "F_anchor_fgm" and elapsed_seconds > TIME_BUDGET["drop_F_after_seconds"]:
        component_status[component_name]["active"] = False
        component_drop_log.append({
            "component": component_name, "stage": "final_training",
            "reason": f"time budget: elapsed {elapsed_seconds/3600:.2f}h > drop_F threshold",
        })
        print(f"{component_name}: dropped on time budget")
        continue

    for seed in component["seeds"]:
        # Reuse the winning-LR seed-30 screening run verbatim instead of
        # retraining. Checked BEFORE the time budget: reuse is free and must
        # never be discarded on a time-constrained run.
        screen_key = (component_name, float(run_config["learning_rate"]))
        if (
            CONFIG["screening"]["reuse_winning_screen_run_as_final_seed_30_run"]
            and seed == SCREEN_SEED
            and screen_key in screen_store
        ):
            stored = screen_store[screen_key]
            val_metrics = compute_binary_metrics(y_val, stored["val_prob"], 0.50)
            run_records.append({
                "component": component_name,
                "seed": seed,
                "val_prob": stored["val_prob"],
                "test_prob": stored["test_prob"],
                "best_epoch": stored["best_epoch"],
                "val_accuracy_at_0_50": val_metrics["accuracy"],
                "reused_from_screen": True,
            })
            history = stored["history"].copy()
            history["member"] = component_name
            final_histories.append(history)
            print(f"{component_name} seed {seed}: reused winning screening run")
            continue

        if component_name == "E_twitter_large":
            elapsed_seconds = time.time() - RUN_START_TIME
            if elapsed_seconds > TIME_BUDGET["stop_launching_E_final_runs_after_seconds"]:
                component_drop_log.append({
                    "component": component_name, "stage": f"final_seed_{seed}",
                    "reason": "time budget: stopped launching E runs",
                })
                print(f"{component_name} seed {seed}: not launched (time budget)")
                continue

        run_label = f"final_{component_name}_seed_{seed}"
        print("\n" + "=" * 90)
        print("Training", run_label)
        result = run_with_degradation(component_name, run_config, run_label, seed)
        if result is None:
            continue

        test_prob_run = compute_test_probs(result, run_config)
        val_metrics = result["best_val_metrics_at_050"]
        run_records.append({
            "component": component_name,
            "seed": seed,
            "val_prob": result["best_val_prob"].copy(),
            "test_prob": test_prob_run,
            "best_epoch": result["best_epoch"],
            "val_accuracy_at_0_50": val_metrics["accuracy"],
            "reused_from_screen": False,
        })
        history = result["training_history"].copy()
        history["member"] = component_name
        final_histories.append(history)

        keep_as_representative = (
            component_name == REPRESENTATIVE_COMPONENT
            and (
                representative_model is None
                or int(seed) == PREFERRED_REPRESENTATIVE_SEED
            )
        )
        if keep_as_representative:
            if representative_model is not None:
                del representative_model
                gc.collect()
            representative_model = result["model"].to("cpu")
            representative_tokenizer = result["tokenizer"]
            representative_seed = int(seed)
        else:
            del result["model"]

        del result
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

component_runs = {}
for record in run_records:
    component_runs.setdefault(record["component"], []).append(record)

final_training_history = (
    pd.concat(final_histories, ignore_index=True) if final_histories else pd.DataFrame()
)
final_training_history.to_csv(
    OUTPUT_DIR / "training_logs" / "final_training_history.csv", index=False
)

final_seed_summary = pd.DataFrame([
    {
        "component": record["component"],
        "random_seed": record["seed"],
        "best_epoch": record["best_epoch"],
        "val_accuracy_at_0_50": record["val_accuracy_at_0_50"],
        "reused_from_screen": record["reused_from_screen"],
    }
    for record in run_records
])
final_seed_summary.to_csv(OUTPUT_DIR / "training_logs" / "final_seed_summary.csv", index=False)

print("\nCompleted runs per component:")
for component_name, records in component_runs.items():
    print(f"  {component_name}: seeds {sorted(r['seed'] for r in records)}")
print("Representative model:", REPRESENTATIVE_COMPONENT, "seed", representative_seed)
display(final_seed_summary)


In [ ]:
# ============================================================
# 15. Admission floors and per-component pooling
# ============================================================

# A component enters the pool iff its seed-pooled validation accuracy at the
# fixed 0.50 operating point clears the floor. The floor (0.84) sits above the
# entire measured-dead lexical family (0.80-0.83) and below every past
# transformer run (>=0.8511): it is a broken-run guard, not a tuner.
ADMISSION_FLOOR = float(CONFIG["pooling"]["admission_floor_val_accuracy_at_0_50"])
MIN_SEEDS = int(CONFIG["pooling"]["min_seeds_for_admission"])

component_pool = {}      # name -> {"val_prob", "test_prob", "val_accuracy", "admitted", ...}
admission_rows = []

for component_name, records in component_runs.items():
    seed_val_matrix = np.vstack([record["val_prob"] for record in records])
    seed_test_matrix = np.vstack([record["test_prob"] for record in records])
    pooled_val_prob = logit_mean_pool(seed_val_matrix)
    pooled_test_prob = logit_mean_pool(seed_test_matrix)
    pooled_val_metrics = compute_binary_metrics(y_val, pooled_val_prob, 0.50)

    enough_seeds = len(records) >= MIN_SEEDS
    admitted = bool(enough_seeds and pooled_val_metrics["accuracy"] >= ADMISSION_FLOOR)
    rejection_reason = ""
    if not enough_seeds:
        rejection_reason = f"only {len(records)} seeds completed (< {MIN_SEEDS})"
    elif not admitted:
        rejection_reason = (
            f"pooled val accuracy {pooled_val_metrics['accuracy']:.4f} < floor {ADMISSION_FLOOR}"
        )

    component_pool[component_name] = {
        "val_prob": pooled_val_prob,
        "test_prob": pooled_test_prob,
        "val_accuracy": pooled_val_metrics["accuracy"],
        "seeds_completed": sorted(record["seed"] for record in records),
        "admitted": admitted,
        "in_primary_pool": bool(CONFIG["components"][component_name]["in_primary_pool"]),
    }
    admission_rows.append({
        "component": component_name,
        "checkpoint": component_status[component_name]["checkpoint"],
        "seeds_completed": len(records),
        "admitted": admitted,
        "in_primary_pool": component_pool[component_name]["in_primary_pool"],
        "rejection_reason": rejection_reason,
        **{f"pooled_val_{key}": value for key, value in pooled_val_metrics.items()},
    })
    print(
        f"{component_name}: pooled val acc@0.50={pooled_val_metrics['accuracy']:.4f} "
        f"({len(records)} seeds) -> admitted={admitted} {rejection_reason}"
    )

admission_table = pd.DataFrame(admission_rows)
admission_table.to_csv(
    OUTPUT_DIR / "auxiliary" / "component_admission_validation.csv", index=False
)

admitted_primary_components = [
    name for name, pool in component_pool.items()
    if pool["admitted"] and pool["in_primary_pool"]
]
if not admitted_primary_components:
    raise RuntimeError(
        "No primary-pool component cleared admission; the run has no valid model to select."
    )
print("\nAdmitted primary-pool components:", admitted_primary_components)


In [ ]:
# ============================================================
# 16. Prespecified primary, gated challengers, anchor fallback, unlock gate
# ============================================================

PRIMARY_THRESHOLD = float(CONFIG["selection"]["prespecified_primary_threshold"])
MIN_GAIN = int(CONFIG["selection"]["minimum_gain_correct_examples_over_primary"])
N_VAL = len(y_val)


def pool_components(component_names, weights=None):
    """Equal-weight (or weighted) mean of component mean-logits -> probability.

    Seeds were already averaged within each component (cell 15), so no
    checkpoint outvotes another regardless of how many seeds it completed.
    """
    names = list(component_names)
    if weights is None:
        weights = [1.0] * len(names)
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    val_logits = np.stack([probability_to_logit(component_pool[n]["val_prob"]) for n in names])
    test_logits = np.stack([probability_to_logit(component_pool[n]["test_prob"]) for n in names])
    pooled_val = sigmoid((weights[:, None] * val_logits).sum(axis=0))
    pooled_test = sigmoid((weights[:, None] * test_logits).sum(axis=0))
    return pooled_val, pooled_test


# --- Prespecified PRIMARY: equal-weight pool of admitted A-E components @ 0.50.
primary_val_prob, primary_test_prob = pool_components(admitted_primary_components)
primary_metrics = compute_binary_metrics(y_val, primary_val_prob, PRIMARY_THRESHOLD)
primary_correct = correct_examples(primary_metrics["accuracy"], N_VAL)

strategy_rows = [{
    "strategy": "primary__equal_weight_pool_A_to_E__fixed_0_50",
    "threshold_policy": "prespecified_fixed",
    "selected_validation_threshold": PRIMARY_THRESHOLD,
    "components": "|".join(admitted_primary_components),
    "status": "evaluated",
    **primary_metrics,
}]
strategy_payloads = {
    "primary__equal_weight_pool_A_to_E__fixed_0_50": (
        primary_val_prob, primary_test_prob, PRIMARY_THRESHOLD
    ),
}

# --- Challengers: exactly four, one scored validation look each.
challenger_definitions = []

if "A_anchor" in admitted_primary_components and len(admitted_primary_components) > 1:
    challenger_definitions.append((
        "C1__anchor_weight_2_logit_mean__fixed_0_50",
        lambda: pool_components(
            admitted_primary_components,
            weights=[2.0 if n == "A_anchor" else 1.0 for n in admitted_primary_components],
        ) + (PRIMARY_THRESHOLD,),
    ))
else:
    strategy_rows.append({
        "strategy": "C1__anchor_weight_2_logit_mean__fixed_0_50",
        "status": "vacated: anchor absent or pool has a single member",
    })

if len(admitted_primary_components) > 3:
    top3 = sorted(
        admitted_primary_components,
        key=lambda n: component_pool[n]["val_accuracy"],
        reverse=True,
    )[:3]
    challenger_definitions.append((
        "C2__top3_components_logit_mean__fixed_0_50",
        lambda: pool_components(top3) + (PRIMARY_THRESHOLD,),
    ))
else:
    strategy_rows.append({
        "strategy": "C2__top3_components_logit_mean__fixed_0_50",
        "status": "vacated: pool already has <= 3 components",
    })

if component_pool.get("F_anchor_fgm", {}).get("admitted", False):
    challenger_definitions.append((
        "C3__primary_pool_plus_fgm__fixed_0_50",
        lambda: pool_components(admitted_primary_components + ["F_anchor_fgm"])
        + (PRIMARY_THRESHOLD,),
    ))
else:
    strategy_rows.append({
        "strategy": "C3__primary_pool_plus_fgm__fixed_0_50",
        "status": "vacated: F not admitted",
    })


def tuned_threshold_challenger():
    # The one tuned-threshold look, with the stability machinery ON
    # (near-optimal tolerance 1 correct example, 200 stratified bootstrap
    # repeats) and its full sweep preserved for diagnostics.
    tuned_threshold, sweep_df, stability = select_stable_threshold(
        y_val,
        primary_val_prob,
        CONFIG["threshold_selection"]["threshold_grid"],
        metric=CONFIG["threshold_selection"]["metric"],
        tolerance_correct_examples=CONFIG["threshold_selection"][
            "near_optimal_tolerance_correct_examples"
        ],
        bootstrap_repeats=CONFIG["threshold_selection"]["bootstrap_repeats"],
        random_seed=CONFIG["random_seed"],
    )
    sweep_df["selected_operating_threshold"] = np.isclose(sweep_df["threshold"], tuned_threshold)
    sweep_df.to_csv(OUTPUT_DIR / "threshold_sweep_validation.csv", index=False)
    save_json(stability, OUTPUT_DIR / "auxiliary" / "tuned_threshold_stability.json")
    return primary_val_prob, primary_test_prob, tuned_threshold


challenger_definitions.append((
    "C4__primary_pool__val_tuned_stable_threshold",
    tuned_threshold_challenger,
))

for strategy_name, build in challenger_definitions:
    challenger_val_prob, challenger_test_prob, challenger_threshold = build()
    challenger_metrics = compute_binary_metrics(y_val, challenger_val_prob, challenger_threshold)
    gain = correct_examples(challenger_metrics["accuracy"], N_VAL) - primary_correct
    strategy_rows.append({
        "strategy": strategy_name,
        "threshold_policy": (
            "validation_tuned_stable" if strategy_name.startswith("C4") else "prespecified_fixed"
        ),
        "selected_validation_threshold": challenger_threshold,
        "status": "evaluated",
        "validation_gain_correct_examples_over_primary": gain,
        **challenger_metrics,
    })
    strategy_payloads[strategy_name] = (
        challenger_val_prob, challenger_test_prob, challenger_threshold
    )

# --- Challenger rule: replace the primary only on a >=3-correct-row gain.
evaluated = [
    row for row in strategy_rows
    if row.get("status") == "evaluated" and not row["strategy"].startswith("primary")
]
qualifying = [
    row for row in evaluated
    if row.get("validation_gain_correct_examples_over_primary", 0) >= MIN_GAIN
]
if qualifying:
    best = max(qualifying, key=lambda row: (row["accuracy"], -evaluated.index(row)))
    selected_strategy_name = best["strategy"]
    selection_reason = (
        f"Challenger {selected_strategy_name} gained "
        f"{best['validation_gain_correct_examples_over_primary']} correct validation "
        f"examples over the prespecified primary (required: {MIN_GAIN})."
    )
else:
    selected_strategy_name = "primary__equal_weight_pool_A_to_E__fixed_0_50"
    best_gain = max(
        (row.get("validation_gain_correct_examples_over_primary", 0) for row in evaluated),
        default=0,
    )
    selection_reason = (
        f"Retained the prespecified primary; the strongest challenger gained only "
        f"{best_gain} correct validation examples, below the required {MIN_GAIN}."
    )

val_prob, test_prob, selected_threshold = strategy_payloads[selected_strategy_name]
selected_metrics = compute_binary_metrics(y_val, val_prob, selected_threshold)

# --- Anchor fallback: never ship a pool that loses to a nb10 reproduction.
anchor_fallback_engaged = False
if "A_anchor" in component_pool:
    anchor_val_prob = component_pool["A_anchor"]["val_prob"]
    anchor_test_prob = component_pool["A_anchor"]["test_prob"]
    anchor_metrics = compute_binary_metrics(y_val, anchor_val_prob, PRIMARY_THRESHOLD)
    anchor_deficit = (
        correct_examples(anchor_metrics["accuracy"], N_VAL)
        - correct_examples(selected_metrics["accuracy"], N_VAL)
    )
    strategy_rows.append({
        "strategy": "reference__anchor_only_pool__fixed_0_50",
        "threshold_policy": "prespecified_fixed",
        "selected_validation_threshold": PRIMARY_THRESHOLD,
        "status": "reference_only",
        **anchor_metrics,
    })
    fallback_margin = int(
        CONFIG["selection"]["fallback_to_anchor_only_if_primary_trails_by_rows"]
    )
    if anchor_deficit >= fallback_margin:
        anchor_fallback_engaged = True
        selected_strategy_name = "anchor_only_fallback__fixed_0_50"
        val_prob, test_prob, selected_threshold = (
            anchor_val_prob, anchor_test_prob, PRIMARY_THRESHOLD
        )
        selected_metrics = anchor_metrics
        selection_reason = (
            f"Anchor fallback engaged: the selected pool trailed the anchor-alone pool by "
            f"{anchor_deficit} validation rows (>= {fallback_margin}). Shipping the "
            "nb10-equivalent anchor; the test unlock is refused for a reproduction."
        )
        # The sweep CSV must carry a row for the shipped strategy so its
        # selected_strategy flag marks exactly one row.
        strategy_rows.append({
            "strategy": selected_strategy_name,
            "threshold_policy": "prespecified_fixed",
            "selected_validation_threshold": PRIMARY_THRESHOLD,
            "status": "fallback_engaged",
            **anchor_metrics,
        })

# --- Test-unlock gate.
gate_config = CONFIG["test_unlock_gate"]
selected_correct = correct_examples(selected_metrics["accuracy"], N_VAL)
gate_threshold_correct = int(gate_config["min_val_correct_of_450"])
gate_enforced = bool(gate_config["enforce_validation_gate"])

if not gate_enforced:
    TEST_UNLOCKED = True
    gate_note = "validation gate DISABLED by CONFIG override; test will be evaluated"
elif anchor_fallback_engaged and gate_config["refuse_unlock_if_selected_is_anchor_only_fallback"]:
    TEST_UNLOCKED = False
    gate_note = "unlock refused: anchor-only fallback (re-measuring nb10 must not spend an unlock)"
elif selected_correct >= gate_threshold_correct:
    TEST_UNLOCKED = True
    gate_note = (
        f"gate passed: {selected_correct}/450 validation correct >= {gate_threshold_correct}"
    )
else:
    TEST_UNLOCKED = False
    gate_note = (
        f"gate failed: {selected_correct}/450 validation correct < {gate_threshold_correct}; "
        "no test metrics will be computed this run"
    )

strategy_results = pd.DataFrame(strategy_rows)
strategy_results["selected_strategy"] = strategy_results["strategy"].eq(selected_strategy_name)
strategy_results.to_csv(
    OUTPUT_DIR / "auxiliary" / "strategy_sweep_validation.csv", index=False
)

validation_metrics = selected_metrics
save_json(validation_metrics, OUTPUT_DIR / "metrics_validation.json")

# best_config.json is written exactly once, here, after every decision is final.
best_config_summary = {
    "technique": TECHNIQUE_NAME,
    "model_family": CONFIG["model_family"],
    "feature_family": CONFIG["feature_family"],
    "dataset_version": CONFIG["dataset_version"],
    "split_version": CONFIG["split_version"],
    "random_seed": CONFIG["random_seed"],
    "component_configurations": {
        name: {
            key: value for key, value in config.items()
            if key not in {"component"}
        }
        for name, config in final_component_configs.items()
    },
    "component_admission": {
        name: {
            "admitted": pool["admitted"],
            "pooled_val_accuracy_at_0_50": pool["val_accuracy"],
            "seeds_completed": pool["seeds_completed"],
        }
        for name, pool in component_pool.items()
    },
    "admitted_primary_components": admitted_primary_components,
    "selected_strategy": selected_strategy_name,
    "selected_threshold": float(selected_threshold),
    "threshold_selected_on_split": "validation",
    "strategy_selection_reason": selection_reason,
    "anchor_fallback_engaged": anchor_fallback_engaged,
    "validation_gate": {**gate_config, "note": gate_note, "selected_correct": selected_correct},
    "test_unlocked": TEST_UNLOCKED,
    "test_set_used_for_selection": False,
    "validation_metrics": validation_metrics,
}
save_json(best_config_summary, OUTPUT_DIR / "best_config.json")

print("Selected strategy:", selected_strategy_name)
print("Selected threshold:", selected_threshold)
print("Selection reason:", selection_reason)
print("Gate:", gate_note)
print(json.dumps(validation_metrics, indent=2))
display(strategy_results[[c for c in strategy_results.columns if c in (
    "strategy", "status", "selected_validation_threshold", "accuracy",
    "balanced_accuracy", "positive_f1",
    "validation_gain_correct_examples_over_primary", "selected_strategy",
)]])


In [ ]:
# ============================================================
# 17. Final held-out test evaluation (single pass, report-only)
# ============================================================

# Every selection decision — configurations, pool membership, strategy, and
# threshold — is already locked on validation. Nothing below feeds back.
if TEST_UNLOCKED:
    test_metrics = compute_binary_metrics(y_test, test_prob, selected_threshold)
    save_json(test_metrics, OUTPUT_DIR / "metrics_test.json")

    print("Held-out test metrics:")
    print(json.dumps(test_metrics, indent=2))

    y_test_pred = (test_prob >= selected_threshold).astype(int)
    report_text = classification_report(y_test, y_test_pred, digits=4, zero_division=0)
    print("\nClassification report:")
    print(report_text)

    report_dict = classification_report(
        y_test, y_test_pred, digits=4, zero_division=0, output_dict=True
    )
    save_json(report_dict, OUTPUT_DIR / "classification_report_test.json")

    # Long-form confusion matrix per docs/RESULTS_SCHEMA.md (actual,predicted,
    # count) with the repo's class names, matching tools/eval_from_probs.py.
    cm = confusion_matrix(y_test, y_test_pred, labels=[0, 1])
    class_names = {0: "non_extremist", 1: "extremist"}
    confusion_long = pd.DataFrame(
        [
            {
                "actual": class_names[actual],
                "predicted": class_names[predicted],
                "count": int(cm[actual, predicted]),
            }
            for actual in (0, 1)
            for predicted in (0, 1)
        ]
    )
    confusion_long.to_csv(OUTPUT_DIR / "confusion_matrix_test.csv", index=False)

    accuracy_target_summary = {
        "test_support": int(len(y_test)),
        "correct_predictions": int((y_test_pred == y_test).sum()),
        "incorrect_predictions": int((y_test_pred != y_test).sum()),
        "accuracy": float(test_metrics["accuracy"]),
        "minimum_correct_for_at_least_90_percent": int(math.ceil(0.90 * len(y_test))),
        "minimum_correct_for_strictly_above_90_percent": int(math.floor(0.90 * len(y_test)) + 1),
    }
    accuracy_target_summary["additional_correct_needed_for_at_least_90_percent"] = max(
        0,
        accuracy_target_summary["minimum_correct_for_at_least_90_percent"]
        - accuracy_target_summary["correct_predictions"],
    )
    save_json(accuracy_target_summary, OUTPUT_DIR / "accuracy_target_summary.json")
    print(json.dumps(accuracy_target_summary, indent=2))
else:
    test_metrics = None
    print(
        "VALIDATION GATE NOT MET: the test split stays locked for this run. "
        "Probability artifacts are still exported for local, gate-respecting analysis."
    )


In [ ]:
# ============================================================
# 18. Sanitized probability artifacts, prediction files, model artifacts
# ============================================================

# --- 1. The committable, text-free probability artifacts (the repo contract).
# These export UNCONDITIONALLY: test probabilities are inference-only numbers,
# and a gate-blocked run must still leave a complete record for local,
# gate-respecting analysis via tools/eval_from_probs.py.
probs_dir = OUTPUT_DIR / "probs"
members_dir = probs_dir / "members"
members_dir.mkdir(parents=True, exist_ok=True)

export_probability_artifact(val_df, val_prob, "validation", probs_dir)
export_probability_artifact(test_df, test_prob, "test", probs_dir)

if CONFIG["probs_export"]["export_per_component_pooled"]:
    for component_name, pool in component_pool.items():
        safe_component = sanitize_name(component_name)
        for split_name, source_df, probabilities in [
            ("validation", val_df, pool["val_prob"]),
            ("test", test_df, pool["test_prob"]),
        ]:
            artifact = pd.DataFrame({
                "row_id": source_df[id_col].astype(str).values,
                "split": split_name,
                "y_true": source_df[label_col].astype(int).values,
                "y_prob": np.asarray(probabilities, dtype=float),
            })
            artifact.to_csv(
                members_dir / f"{TECHNIQUE_NAME}__component-{safe_component}__{split_name}.csv",
                index=False,
            )

if CONFIG["probs_export"]["export_per_run"]:
    for record in run_records:
        safe_component = sanitize_name(record["component"])
        for split_name, source_df, probabilities in [
            ("validation", val_df, record["val_prob"]),
            ("test", test_df, record["test_prob"]),
        ]:
            artifact = pd.DataFrame({
                "row_id": source_df[id_col].astype(str).values,
                "split": split_name,
                "y_true": source_df[label_col].astype(int).values,
                "y_prob": np.asarray(probabilities, dtype=float),
            })
            artifact.to_csv(
                members_dir
                / f"{TECHNIQUE_NAME}__member-{safe_component}-seed{record['seed']}__{split_name}.csv",
                index=False,
            )

probs_meta = {
    "technique": TECHNIQUE_NAME,
    "run_id": RUN_ID,
    "model_family": CONFIG["model_family"],
    "feature_family": CONFIG["feature_family"],
    "dataset_version": CONFIG["dataset_version"],
    "split_version": CONFIG["split_version"],
    "random_seed": CONFIG["random_seed"],
    "hyperparameters": {
        name: {
            "checkpoint": config["checkpoint"],
            "learning_rate": config["learning_rate"],
            "num_epochs": config["num_epochs"],
            "batch_size": config["batch_size"],
            "gradient_accumulation_steps": config["gradient_accumulation_steps"],
            "max_length": config["max_length"],
            "weight_decay": config["weight_decay"],
            "class_weight": config["class_weight"],
            "label_smoothing": config["label_smoothing"],
            "use_fgm": config["use_fgm"],
            "amp_dtype": config["amp_dtype"],
        }
        for name, config in final_component_configs.items()
    },
    "selected_strategy": selected_strategy_name,
    "selected_threshold": float(selected_threshold),
    "threshold_selected_on_split": "validation",
    "gate": {"test_unlocked": TEST_UNLOCKED, "note": gate_note},
    "contains_raw_text": False,
    "row_counts": {
        "validation": {"total": int(len(val_df)), "positive": int(y_val.sum())},
        "test": {"total": int(len(test_df)), "positive": int(y_test.sum())},
    },
    "validation_looks_declared": CONFIG["validation_budget_declared"],
    "schema": CONFIG["probs_export"]["required_columns"],
}
save_json(probs_meta, probs_dir / f"{TECHNIQUE_NAME}__meta.json")
print("Probability artifacts written to:", probs_dir)

# --- 2. Rich per-row prediction frames (Kaggle-side ONLY: they contain raw text).
component_val_probabilities = {
    f"component_{sanitize_name(name)}_prob": pool["val_prob"]
    for name, pool in component_pool.items()
}
component_test_probabilities = {
    f"component_{sanitize_name(name)}_prob": pool["test_prob"]
    for name, pool in component_pool.items()
}

validation_predictions = make_prediction_frame(
    val_df, val_prob, selected_threshold, component_val_probabilities
)
validation_predictions["selected_strategy"] = selected_strategy_name
validation_predictions.to_csv(OUTPUT_DIR / "predictions_validation.csv", index=False)

if TEST_UNLOCKED:
    test_predictions = make_prediction_frame(
        test_df, test_prob, selected_threshold, component_test_probabilities
    )
    test_predictions["selected_strategy"] = selected_strategy_name
    test_predictions.to_csv(OUTPUT_DIR / "predictions_test.csv", index=False)
else:
    test_predictions = None
    print("Gate not met: predictions_test.csv is not written (no test outcome may be read).")

# XAI and error analysis operate on the test frame when unlocked, otherwise on
# validation so the interpretability deliverable survives a gate-blocked run.
if TEST_UNLOCKED:
    xai_frame = test_predictions
    xai_split_name = "test"
else:
    xai_frame = validation_predictions
    xai_split_name = "validation"
print("Interpretability / error-analysis split:", xai_split_name)

# --- 3. Representative model artifacts and XAI handles. A missing
# representative degrades XAI/model export but must never kill the closing
# chain (probability export, metadata, zip) of an otherwise-valid pool run.
if representative_model is not None:
    model_dir = OUTPUT_DIR / "model_artifacts" / "transformer_model"
    model_dir.mkdir(parents=True, exist_ok=True)
    representative_model.save_pretrained(model_dir)
    representative_tokenizer.save_pretrained(model_dir)
    xai_model = representative_model.to(DEVICE)
    xai_tokenizer = representative_tokenizer
    print("Saved representative transformer model/tokenizer:", model_dir)
else:
    xai_model = None
    xai_tokenizer = None
    print(
        "WARNING: no representative anchor model was retained; "
        "model export and gradient attribution are skipped."
    )

joblib.dump(
    {
        "technique_name": TECHNIQUE_NAME,
        "selected_threshold": float(selected_threshold),
        "selected_strategy": selected_strategy_name,
        "component_configurations": {
            name: config["checkpoint"] for name, config in final_component_configs.items()
        },
        "admitted_primary_components": admitted_primary_components,
        "representative_component": CONFIG["explainability"]["representative_component"],
        "representative_seed": representative_seed,
        "text_normalization_mode": CONFIG["text_normalization_mode"],
    },
    OUTPUT_DIR / "model_artifacts" / "inference_metadata.joblib",
)

print("Saved validation predictions:", OUTPUT_DIR / "predictions_validation.csv")


In [ ]:
# ============================================================
# 19. Evaluation plots
# ============================================================

def save_confusion_matrix_plot(y_true, y_prob, threshold, path, title):
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(cm)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["pred_0", "pred_1"])
    ax.set_yticklabels(["true_0", "true_1"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    fig.tight_layout(); fig.savefig(path, dpi=200, bbox_inches="tight"); plt.show()


def save_roc_plot(y_true, y_prob, path, title):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, label=f"ROC-AUC = {safe_roc_auc(y_true, y_prob):.4f}")
    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title(title); ax.legend(); fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight"); plt.show()


def save_pr_plot(y_true, y_prob, path, title):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(recall, precision, label=f"PR-AUC = {safe_pr_auc(y_true, y_prob):.4f}")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title(title); ax.legend(); fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight"); plt.show()


def save_calibration_plot(y_true, y_prob, path, title):
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy="uniform")
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(prob_pred, prob_true, marker="o")
    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Fraction of positives")
    ax.set_title(title); fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight"); plt.show()


if TEST_UNLOCKED:
    save_confusion_matrix_plot(
        y_test, test_prob, selected_threshold,
        OUTPUT_DIR / "plots" / "confusion_matrix_test.png",
        f"{TECHNIQUE_NAME} test confusion matrix",
    )
    save_roc_plot(
        y_test, test_prob,
        OUTPUT_DIR / "plots" / "roc_curve_test.png",
        f"{TECHNIQUE_NAME} test ROC",
    )
    save_pr_plot(
        y_test, test_prob,
        OUTPUT_DIR / "plots" / "pr_curve_test.png",
        f"{TECHNIQUE_NAME} test precision-recall",
    )
    save_calibration_plot(
        y_test, test_prob,
        OUTPUT_DIR / "plots" / "calibration_curve_test.png",
        f"{TECHNIQUE_NAME} test calibration",
    )
else:
    print("Validation gate not met: test plots are skipped.")

if not final_training_history.empty:
    fig, ax = plt.subplots(figsize=(9, 5))
    for (member, seed), run_history in final_training_history.groupby(["member", "random_seed"]):
        ax.plot(
            run_history["epoch"], run_history["train_loss_mean"],
            marker="o", label=f"{member} seed {seed}",
        )
    ax.set_xlabel("Epoch"); ax.set_ylabel("Mean training loss")
    ax.set_title(f"{TECHNIQUE_NAME} final training loss")
    ax.legend(fontsize=7, ncol=2)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "plots" / "final_training_loss.png", dpi=200, bbox_inches="tight")
    plt.show()


In [ ]:
# ============================================================
# 20. Global interpretability: Gradient x Embedding
# ============================================================

# Explanations use one representative fine-tuned run (declared in CONFIG).
# Attribution scores explain that single transformer, not the pooled ensemble;
# they are model-behavior signals, not proof of true reasoning (docs/MODEL_CARD.md).
SPECIAL_TOKEN_SET = (
    set(xai_tokenizer.all_special_tokens) if xai_tokenizer is not None else set()
)


def clean_token_for_display(token):
    return token.replace("Ġ", "").replace("▁", "")


def gradient_x_embedding_attribution(model, tokenizer, text, target_class=1, max_length=192):
    model.eval()
    encoded = tokenizer(
        str(text), truncation=True, max_length=int(max_length), return_tensors="pt"
    )
    input_ids = encoded["input_ids"].to(DEVICE)
    model_inputs = {
        key: value.to(DEVICE) for key, value in encoded.items() if key != "input_ids"
    }

    # Detach to create a leaf tensor whose gradient can be read reliably.
    embeddings = model.get_input_embeddings()(input_ids).detach()
    embeddings.requires_grad_(True)

    model.zero_grad(set_to_none=True)
    outputs = model(inputs_embeds=embeddings, **model_inputs)
    probabilities = torch.softmax(outputs.logits, dim=-1)[0]
    score = outputs.logits[0, int(target_class)]
    score.backward()

    attribution_values = (
        embeddings.grad * embeddings
    ).sum(dim=-1).squeeze(0).detach().cpu().numpy()
    token_ids = input_ids.squeeze(0).detach().cpu().numpy().tolist()
    tokens = tokenizer.convert_ids_to_tokens(token_ids)

    records = []
    for position, (token, token_id, attribution) in enumerate(
        zip(tokens, token_ids, attribution_values)
    ):
        if token in SPECIAL_TOKEN_SET:
            continue
        records.append({
            "token_position": position,
            "token": token,
            "token_display": clean_token_for_display(token),
            "token_id": int(token_id),
            "attribution_to_positive_logit": float(attribution),
            "abs_attribution_to_positive_logit": float(abs(attribution)),
            "representative_transformer_positive_probability": float(
                probabilities[1].detach().cpu()
            ),
        })
    return records


attr_sample = xai_frame.copy()
attr_sample["priority"] = attr_sample["error_type"].isin(
    ["false_positive", "false_negative"]
).astype(int)
attr_sample["confidence_distance"] = np.abs(attr_sample["y_prob"] - selected_threshold)
attr_sample = attr_sample.sort_values(
    ["priority", "confidence_distance"], ascending=[False, False]
).head(CONFIG["explainability"]["max_global_attribution_examples"])

SKIP_XAI = xai_model is None or (
    time.time() - RUN_START_TIME
    > CONFIG["time_budget"]["skip_xai_and_trim_error_analysis_after_seconds"]
)
if SKIP_XAI:
    attr_sample = attr_sample.head(0)
    print("Attribution skipped (no representative model or time budget exceeded).")

global_token_rows = []
for _, row in attr_sample.iterrows():
    try:
        records = gradient_x_embedding_attribution(
            xai_model,
            xai_tokenizer,
            row["model_input_text"],
            target_class=CONFIG["positive_label"],
            max_length=CONFIG["max_length"],
        )
        for record in records:
            record.update({
                "row_id": row["row_id"],
                "y_true": row["y_true"],
                "y_pred": row["y_pred"],
                "final_pooled_probability": row["y_prob"],
                "error_type": row["error_type"],
            })
            global_token_rows.append(record)
    except RuntimeError as error:
        print(f"Attribution skipped for row_id={row['row_id']}: {error}")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

global_token_attributions = pd.DataFrame(global_token_rows)
if not global_token_attributions.empty:
    global_summary = (
        global_token_attributions.groupby("token_display", as_index=False)
        .agg(
            mean_attribution_to_positive_logit=("attribution_to_positive_logit", "mean"),
            mean_abs_attribution_to_positive_logit=("abs_attribution_to_positive_logit", "mean"),
            occurrence_count=("token_display", "size"),
        )
        .sort_values("mean_attribution_to_positive_logit", ascending=False)
    )
    global_token_attributions.to_csv(
        OUTPUT_DIR / "interpretability" / "global_token_attributions_long.csv", index=False
    )
    global_summary.to_csv(
        OUTPUT_DIR / "interpretability" / "global_token_attribution_summary.csv", index=False
    )
    global_summary.head(CONFIG["explainability"]["top_n_global"]).to_csv(
        OUTPUT_DIR / "interpretability" / "top_positive_tokens_by_gradient.csv", index=False
    )
    global_summary.tail(CONFIG["explainability"]["top_n_global"]).sort_values(
        "mean_attribution_to_positive_logit", ascending=True
    ).to_csv(
        OUTPUT_DIR / "interpretability" / "top_negative_tokens_by_gradient.csv", index=False
    )
    print("Global attribution rows:", len(global_token_attributions))
    print("Distinct tokens:", int(global_summary.shape[0]))
else:
    global_summary = pd.DataFrame()
    print("No global attribution rows were produced.")


In [ ]:
# ============================================================
# 21. Local interpretability for selected examples (row_id-keyed)
# ============================================================

local_parts = []
n_local = CONFIG["explainability"]["local_examples_per_bucket"]
local_parts.append(
    xai_frame[xai_frame["error_type"] == "false_positive"]
    .sort_values("y_prob", ascending=False).head(n_local)
    .assign(explanation_bucket="highest_confidence_false_positive")
)
local_parts.append(
    xai_frame[xai_frame["error_type"] == "false_negative"]
    .sort_values("y_prob", ascending=True).head(n_local)
    .assign(explanation_bucket="highest_confidence_false_negative")
)
local_parts.append(
    xai_frame[(xai_frame["y_true"] == 1) & xai_frame["correct"]]
    .sort_values("y_prob", ascending=False).head(n_local)
    .assign(explanation_bucket="highest_confidence_true_positive")
)
local_parts.append(
    xai_frame[(xai_frame["y_true"] == 0) & xai_frame["correct"]]
    .sort_values("y_prob", ascending=True).head(n_local)
    .assign(explanation_bucket="highest_confidence_true_negative")
)

local_examples = pd.concat(local_parts, ignore_index=True).drop_duplicates("row_id")
if SKIP_XAI:
    local_examples = local_examples.head(0)
    print("Time budget exceeded: local attribution trimmed to zero examples.")
local_records = []
local_long_rows = []

for _, row in local_examples.iterrows():
    try:
        token_records = gradient_x_embedding_attribution(
            xai_model,
            xai_tokenizer,
            row["model_input_text"],
            target_class=CONFIG["positive_label"],
            max_length=CONFIG["max_length"],
        )
        token_df = pd.DataFrame(token_records)
        if token_df.empty:
            top_positive, top_negative = [], []
        else:
            token_df["row_id"] = row["row_id"]
            token_df["explanation_bucket"] = row["explanation_bucket"]
            local_long_rows.append(token_df)
            top_positive = (
                token_df.sort_values("attribution_to_positive_logit", ascending=False)
                .head(CONFIG["explainability"]["top_n_local_tokens"])
                [["token_display", "attribution_to_positive_logit"]]
                .to_dict("records")
            )
            top_negative = (
                token_df.sort_values("attribution_to_positive_logit", ascending=True)
                .head(CONFIG["explainability"]["top_n_local_tokens"])
                [["token_display", "attribution_to_positive_logit"]]
                .to_dict("records")
            )

        # row_id-keyed only: no text preview column, so the file stays committable
        # after a leakage review of the token fragments (which are 🟠-class, not raw rows).
        local_records.append({
            "row_id": row["row_id"],
            "explanation_bucket": row["explanation_bucket"],
            "y_true": row["y_true"],
            "y_pred": row["y_pred"],
            "y_prob": row["y_prob"],
            "error_type": row["error_type"],
            "top_positive_tokens_json": json.dumps(make_json_safe(top_positive)),
            "top_negative_tokens_json": json.dumps(make_json_safe(top_negative)),
        })
    except RuntimeError as error:
        print(f"Local attribution skipped for row_id={row['row_id']}: {error}")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

local_explanations = pd.DataFrame(local_records)
local_explanations.to_csv(
    OUTPUT_DIR / "interpretability" / "local_token_attribution_explanations.csv", index=False
)
if local_long_rows:
    pd.concat(local_long_rows, ignore_index=True).to_csv(
        OUTPUT_DIR / "interpretability" / "local_token_attributions_long.csv", index=False
    )
print("Local explanation rows:", len(local_explanations))


In [ ]:
# ============================================================
# 22. Comparable, deduplicated error-analysis outputs (row_id-keyed)
# ============================================================

# Sanitized error analysis: no text preview and no raw-text column. Rows are
# keyed by row_id so a reviewer joins back to text locally via the frozen split.
error_df = xai_frame.drop(columns=["text", "model_input_text"]).copy()
error_df["abs_distance_to_threshold"] = np.abs(error_df["y_prob"] - error_df["threshold"])


def confidence_bucket(row):
    if row["abs_distance_to_threshold"] <= CONFIG["error_analysis"]["near_threshold_margin"]:
        return "near_threshold"
    if (
        row["y_prob"] >= CONFIG["error_analysis"]["high_confidence_positive"]
        or row["y_prob"] <= CONFIG["error_analysis"]["high_confidence_negative"]
    ):
        return "high_confidence"
    return "moderate_confidence"


error_df["confidence_bucket"] = error_df.apply(confidence_bucket, axis=1)
error_df["manual_error_category"] = ""
error_df["manual_notes"] = ""

if not local_explanations.empty:
    explanation_cols = local_explanations[
        ["row_id", "top_positive_tokens_json", "top_negative_tokens_json"]
    ].drop_duplicates("row_id")
    error_df = error_df.merge(explanation_cols, on="row_id", how="left")
else:
    error_df["top_positive_tokens_json"] = np.nan
    error_df["top_negative_tokens_json"] = np.nan

# Legacy column order first so downstream comparison scripts keep working
# (text columns intentionally absent; text_hash retained for local joins).
legacy_error_columns = [
    "row_id", "y_true", "split", "y_prob", "threshold", "y_pred",
    "correct", "error_type", "text_hash", "abs_distance_to_threshold",
    "confidence_bucket", "manual_error_category", "manual_notes",
    "top_positive_tokens_json", "top_negative_tokens_json",
]
extra_error_columns = [
    column for column in error_df.columns if column not in legacy_error_columns
]
error_df = error_df[legacy_error_columns + extra_error_columns]
error_df.to_csv(OUTPUT_DIR / "error_analysis" / f"error_analysis_{xai_split_name}.csv", index=False)

# Each error receives one exclusive bucket; no row can appear twice.
errors_only = error_df[error_df["error_type"] != "correct"].copy()
errors_only["review_bucket"] = (
    errors_only["confidence_bucket"] + "_" + errors_only["error_type"]
)
errors_only["review_priority"] = errors_only["confidence_bucket"].map({
    "high_confidence": 1,
    "near_threshold": 2,
    "moderate_confidence": 3,
}).fillna(4)
errors_only["error_confidence_score"] = np.where(
    errors_only["error_type"] == "false_positive",
    errors_only["y_prob"],
    1.0 - errors_only["y_prob"],
)

n_review = CONFIG["error_analysis"]["examples_per_bucket"]
review_df = (
    errors_only.sort_values(
        ["review_priority", "error_confidence_score", "abs_distance_to_threshold"],
        ascending=[True, False, False],
    )
    .groupby("review_bucket", group_keys=False)
    .head(n_review)
    .drop_duplicates("row_id")
    .reset_index(drop=True)
)
review_df.to_csv(OUTPUT_DIR / "error_analysis" / f"manual_review_queue_{xai_split_name}.csv", index=False)

# Preserve the exact three-column summary contract.
error_summary = (
    error_df.groupby(["error_type", "confidence_bucket"])
    .size().reset_index(name="count")
    .sort_values(["error_type", "confidence_bucket"])
)
error_summary.to_csv(OUTPUT_DIR / "error_analysis" / "error_summary.csv", index=False)

manual_category_guide = pd.DataFrame({
    "manual_error_category": [
        "generic_toxicity", "quoted_or_reported_content", "counterspeech",
        "literal_nonhuman_violence", "implicit_extremism", "preprocessing_corruption",
        "ambiguous_label", "likely_label_error", "other",
    ],
    "definition": [
        "Abusive/profane content without the dataset's intended extremist signal.",
        "Extremist content is quoted, described, or reported rather than endorsed.",
        "The author rejects, challenges, or condemns extremist/hateful content.",
        "Violent words refer literally to animals, games, fiction, or another non-target context.",
        "Positive example uses coded, ideological, or contextual cues with little overt toxicity.",
        "Upstream normalization appears to have changed or damaged the wording.",
        "Reasonable annotators could disagree under the present ontology.",
        "The supplied gold label is probably incorrect after adjudication.",
        "Does not fit another category.",
    ],
})
manual_category_guide.to_csv(
    OUTPUT_DIR / "error_analysis" / "manual_error_category_guide.csv", index=False
)

print("Manual-review rows:", len(review_df))
print("Unique manual-review row_ids:", review_df["row_id"].nunique())
display(error_summary)


In [ ]:
# ============================================================
# 23. Experiment metadata, README, and statistical interpretation
# ============================================================

try:
    import transformers
    transformers_version = transformers.__version__
except Exception:
    transformers_version = None

metadata = {
    "run_id": RUN_ID,
    "technique": TECHNIQUE_NAME,
    "created_at_utc": dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z",
    "project_name": CONFIG["project_name"],
    "dataset_version": CONFIG["dataset_version"],
    "split_version": CONFIG["split_version"],
    "random_seed": CONFIG["random_seed"],
    "processed_dataset_path": str(processed_dataset_path),
    "split_assignments_path": str(split_assignments_path),
    "input_file_sha256_16": input_file_hashes,
    "dataset_manifest": dataset_manifest,
    "text_input": text_source_audit,
    "software_environment": {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "torch_version": torch.__version__,
        "transformers_version": transformers_version,
        "cuda_available": torch.cuda.is_available(),
        "cuda_device": (
            torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
        ),
    },
    "component_status": component_status,
    "component_drop_log": component_drop_log,
    "orientation_swap_needed": orientation_swap_needed,
    "model_selection": {
        "screening_split": "validation",
        "screening_rank_metric": CONFIG["screening"]["rank_metric"],
        "epoch_selection_metric": CONFIG["transformer_training"]["metric_for_best_epoch"],
        "admission_floor_val_accuracy_at_0_50": CONFIG["pooling"][
            "admission_floor_val_accuracy_at_0_50"
        ],
        "admitted_primary_components": admitted_primary_components,
        "prespecified_primary": CONFIG["selection"]["prespecified_primary"],
        "selected_strategy": selected_strategy_name,
        "selected_threshold": float(selected_threshold),
        "strategy_selection_reason": selection_reason,
        "anchor_fallback_engaged": anchor_fallback_engaged,
        "threshold_selected_on": "validation",
        "test_set_used_for_selection": False,
        "validation_consumption_ledger": CONFIG["validation_budget_declared"],
    },
    "validation_gate": {
        **CONFIG["test_unlock_gate"],
        "note": gate_note,
        "test_unlocked": TEST_UNLOCKED,
    },
    "final_evaluation": {
        "validation_metrics": validation_metrics,
        "test_metrics": test_metrics,
        "family_position_note": (
            "Approximately the program's 10th test unlock (01-07, 08, 10 precede it; "
            "the reconstructed ledger starts empty). Comparisons must pass "
            "--family-size explicitly; confirm the count against artifacts before running."
        ),
    },
    "xai_method": {
        "global": "Gradient x Embedding token attribution over a capped example sample",
        "local": "Gradient x Embedding for selected examples",
        "split_used": xai_split_name,
        "scope": (
            "Representative anchor seed only; not a complete explanation of the "
            "pooled ensemble prediction."
        ),
    },
    "runtime": {
        "total_wall_clock_hours_at_metadata_cell": round(elapsed_hours(), 3),
        "time_budget": CONFIG["time_budget"],
    },
    "outputs": {
        "config": "config.json",
        "best_config": "best_config.json",
        "ablation_results": "ablation/ablation_results.csv",
        "component_admission": "auxiliary/component_admission_validation.csv",
        "strategy_sweep": "auxiliary/strategy_sweep_validation.csv",
        "metrics_validation": "metrics_validation.json",
        "metrics_test": "metrics_test.json" if TEST_UNLOCKED else None,
        "probability_artifacts": "probs/",
        "predictions_validation": "predictions_validation.csv",
        "predictions_test": "predictions_test.csv" if TEST_UNLOCKED else None,
        "model_artifacts": "model_artifacts/",
        "plots": "plots/",
        "interpretability": "interpretability/",
        "error_analysis": "error_analysis/",
        "training_logs": "training_logs/",
        "data_quality": "data_quality/",
    },
}
save_json(metadata, OUTPUT_DIR / "metadata.json")

# ------------------------------------------------------------------
# Statistical interpretation. This cell issues NO verdict: on this repository
# tools/compare_techniques.py is the sole verdict authority. It restates the
# preregistered framing so the numbers cannot be over-read.
# ------------------------------------------------------------------
stats = CONFIG["statistics"]
interpretation_lines = [
    "STATISTICAL INTERPRETATION (no verdict issued here)",
    f"- Comparator: {stats['comparator']} (test accuracy 0.8889, 400/450).",
    f"- Family size for Holm correction: {stats['family_size_for_holm']} "
    "(confirm against research_loop/test_ledger.jsonl and committed artifacts before comparing).",
    f"- Exact McNemar detectability at n=450: |b-c| >= {stats['mcnemar_unadjusted_min_b_minus_c']} "
    f"unadjusted, |b-c| >= {stats['mcnemar_holm_min_b_minus_c']} Holm-corrected.",
    "- The measured validation->test offset (+2.83pp, sd 0.38, n=7) is a property of the split.",
]
if not TEST_UNLOCKED:
    interpretation_lines.append(
        "- OUTCOME: gate-blocked. No test metric was computed; the probability artifacts "
        "allow a deliberate, ledger-recorded unlock later. This is a legitimate, "
        "publishable non-unlock."
    )
else:
    test_accuracy = float(test_metrics["accuracy"])
    test_correct = int(round(test_accuracy * len(y_test)))
    interpretation_lines.append(
        f"- Test accuracy: {test_accuracy:.4f} ({test_correct}/450) at threshold "
        f"{float(selected_threshold):.3f}."
    )
    if test_accuracy >= 0.9333:
        interpretation_lines.append(
            "- The result is at or above the Holm-corrected detectability bar; even so, the "
            "verdict belongs to tools/compare_techniques.py alone. Note the prereg's warning "
            "that this region sits at or above the plausible annotation-ambiguity floor."
        )
    elif test_accuracy >= 0.90:
        interpretation_lines.append(
            f"- Preregistered reading for a result in [0.9000, 0.9333): "
            f"{stats['mandated_inconclusive_phrase']}. The committed champion is unchanged "
            "unless tools/compare_techniques.py rules otherwise."
        )
    else:
        interpretation_lines.append(
            "- Below the 0.90 target. Differences of this size against notebooks 07/10 are "
            "not statistically detectable at n=450; the expected verdict is INCONCLUSIVE."
        )
interpretation_text = "\n".join(interpretation_lines)
print(interpretation_text)
with open(OUTPUT_DIR / "statistical_interpretation.txt", "w", encoding="utf-8") as f:
    f.write(interpretation_text + "\n")

readme_text = f"""# {TECHNIQUE_NAME} outputs

Standardized outputs for the `{TECHNIQUE_NAME}` experiment: a heterogeneous
checkpoint ensemble (fine-tuning lineage, pretraining corpus, architecture/
tokenizer, and scale diversity) pooled by mean log-odds at a prespecified 0.50
operating point, anchored on notebook 10's exact recipe.

## Decision protocol (all on train+validation)

- Per-component learning-rate screening at the fixed 0.50 operating point.
- Admission floor: pooled validation accuracy at 0.50 >= 0.84 (broken-run guard).
- Prespecified primary: equal-weight mean of admitted components' seed-mean
  logits at threshold 0.50.
- Four challengers, each evaluated once; replacement requires >= 3 correct
  validation examples.
- Anchor fallback if the pool loses to the anchor-alone pool by >= 3 rows
  (the fallback refuses the test unlock).
- Test unlock gate: >= 392/450 validation correct. Gate status this run:
  {gate_note}

## Selected configuration

- Strategy: `{selected_strategy_name}`
- Threshold: `{float(selected_threshold):.3f}`
- Admitted components: `{admitted_primary_components}`
- Representative XAI seed: `{CONFIG['explainability']['representative_component']}`
  seed `{representative_seed}` (split: {xai_split_name})

## Repository-facing artifacts

- `probs/{TECHNIQUE_NAME}__validation.csv`, `probs/{TECHNIQUE_NAME}__test.csv`
  (+ per-component/per-run files under `probs/members/`, and `__meta.json`):
  columns exactly `row_id, split, y_true, y_prob`; no text. Copy into
  `research_loop/probs/` and derive the results folder with
  `python3 tools/eval_from_probs.py --technique {TECHNIQUE_NAME} --threshold {float(selected_threshold)!r}`.
  The exact selected threshold is recorded in `best_config.json` and
  `probs/{TECHNIQUE_NAME}__meta.json` (key `selected_threshold`) — use that
  value verbatim; never retype a rounded one.
- Copy `ablation/ablation_results.csv` into
  `results_summary/{TECHNIQUE_NAME}/` as well: RESULTS_SCHEMA requires it and
  `eval_from_probs.py` does not derive it.
- Everything else in this folder (predictions, model weights, error analysis)
  is Kaggle-side material; files containing raw text must not be committed.

## Interpretation

See `statistical_interpretation.txt`. This run issues no verdict;
`tools/compare_techniques.py` is the sole verdict authority.
"""
with open(OUTPUT_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(readme_text)

print("\nSaved metadata and README to:", OUTPUT_DIR)


In [ ]:
# ============================================================
# 24. Zip outputs for download / external storage
# ============================================================

zip_base = Path("/kaggle/working") / f"{TECHNIQUE_NAME}_outputs"
zip_path = shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=OUTPUT_DIR,
)

print("Created zip file:")
print(zip_path)
print("""
After downloading, back in the repository:
  1. Copy probs/*.csv and probs/*__meta.json into research_loop/probs/
     (probs/members/* too if you want the per-run artifacts for CPU-local
     ensemble research).
  2. Derive the committable results folder locally, using the EXACT threshold
     recorded in best_config.json / probs/*__meta.json (printed below verbatim):
       python3 tools/eval_from_probs.py --technique %s --threshold %r
  3. Copy ablation/ablation_results.csv into results_summary/%s/
     (RESULTS_SCHEMA requires it; eval_from_probs does not derive it).
  4. Validate: python3 tools/validate_results_folder.py --technique %s
  5. Compare (sole verdict authority; --cycle is required, --prereg-sha only if
     a preregistration exists for this run; confirm family size against the
     ledger and committed artifacts first):
       python3 tools/compare_techniques.py --candidate %s \\
         --champion 07_TWITTER-ROBERTA_FINE-TUNE --cycle 11 --family-size 10
Model weights and every text-bearing file stay in this zip / external storage;
do not commit them (docs/RELEASE_CHECKLIST.md).
""" % (TECHNIQUE_NAME, float(selected_threshold), TECHNIQUE_NAME, TECHNIQUE_NAME, TECHNIQUE_NAME))
print("Total wall-clock hours:", round(elapsed_hours(), 2))
